In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from peft import PeftModel
from transformers import AutoTokenizer, AutoConfig, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [3]:
#!pip install streamlit

In [4]:
MODEL_DIR = "/content/drive/MyDrive/B7_light_best_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
with open(os.path.join(MODEL_DIR, "export_meta.json"), "r", encoding="utf-8") as f:
    META = json.load(f)

MODEL_NAME = META["model_name"]
MAX_LENGTH = META["max_length"]
HEAD_DROPOUT = META["head_dropout"]

TARGET_COLS = META["target_cols"]

TR_FEATURE_COLS = META["tr_feature_cols"]
CC_FEATURE_COLS = META["cc_feature_cols"]
LR_FEATURE_COLS = META["lr_feature_cols"]
GRA_FEATURE_COLS = META["gra_feature_cols"]

tr_feat_mean = np.array(META["tr_feat_mean"], dtype=np.float32)
tr_feat_std = np.array(META["tr_feat_std"], dtype=np.float32)

cc_feat_mean = np.array(META["cc_feat_mean"], dtype=np.float32)
cc_feat_std = np.array(META["cc_feat_std"], dtype=np.float32)

lr_feat_mean = np.array(META["lr_feat_mean"], dtype=np.float32)
lr_feat_std = np.array(META["lr_feat_std"], dtype=np.float32)

gra_feat_mean = np.array(META["gra_feat_mean"], dtype=np.float32)
gra_feat_std = np.array(META["gra_feat_std"], dtype=np.float32)

In [6]:
STOPWORDS = {
    "a","an","the","and","or","but","if","while","is","am","are","was","were",
    "be","been","being","of","to","in","on","for","with","as","at","by","from",
    "that","this","these","those","it","its","he","she","they","them","their",
    "we","our","you","your","i","me","my","mine","his","her","hers","do","does",
    "did","have","has","had","will","would","can","could","should","may","might",
    "not","so","than","then","there","here","about","into","over","after","before",
    "more","most","some","any","such","no","nor","too","very"
}

DISCOURSE_MARKERS = [
    "however", "therefore", "moreover", "furthermore", "in addition",
    "for example", "for instance", "on the one hand", "on the other hand",
    "in conclusion", "to conclude", "in summary", "as a result",
    "firstly", "secondly", "finally", "besides", "nevertheless",
    "thus", "overall", "in contrast", "for this reason"
]

LONG_WORD_MIN_LEN = 7
SCORE_MIN = 4.0
SCORE_MAX = 9.0


def safe_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip()


def normalize_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_words(text):
    text = normalize_text(text)
    return re.findall(r"[a-zA-Z']+", text)


def split_sentences(text):
    text = safe_text(text).strip()
    if not text:
        return []
    sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents


def split_paragraphs(text):
    text = safe_text(text)
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paras) == 0 and text.strip():
        paras = [text.strip()]
    return paras


def word_count(text):
    return len(tokenize_words(text))


def unique_ratio(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / len(words)


def root_ttr(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / math.sqrt(len(words))


def repetition_ratio(words):
    if len(words) <= 1:
        return 0.0
    c = Counter(words)
    repeated = sum(v for v in c.values() if v > 1)
    return repeated / len(words)


def repeated_word_ratio(words):
    if len(words) <= 1:
        return 0.0
    repeated_pairs = 0
    for i in range(1, len(words)):
        if words[i] == words[i - 1]:
            repeated_pairs += 1
    return repeated_pairs / len(words)


def avg_word_len(words):
    if len(words) == 0:
        return 0.0
    return float(np.mean([len(w) for w in words]))


def lexical_density_proxy(words):
    if len(words) == 0:
        return 0.0
    content_like = [w for w in words if len(w) > 3 and w not in STOPWORDS]
    return len(content_like) / len(words)


def long_word_ratio(words, min_len=LONG_WORD_MIN_LEN):
    if len(words) == 0:
        return 0.0
    return sum(len(w) >= min_len for w in words) / len(words)


def sentence_lengths(sentences):
    return [len(tokenize_words(s)) for s in sentences if len(tokenize_words(s)) > 0]


def count_discourse_markers(text):
    low = normalize_text(text)
    found = 0
    found_types = 0
    for m in DISCOURSE_MARKERS:
        c = low.count(m)
        if c > 0:
            found += c
            found_types += 1
    return found, found_types


def prompt_keywords(prompt):
    words = tokenize_words(prompt)
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return set(words)


def jaccard_coverage(prompt, essay):
    pk = prompt_keywords(prompt)
    ew = set(tokenize_words(essay))
    if len(pk) == 0:
        return 0.0
    return len(pk & ew) / len(pk)


def prompt_essay_similarity(prompt, essay):
    pw = prompt_keywords(prompt)
    ew = set([w for w in tokenize_words(essay) if w not in STOPWORDS and len(w) > 2])
    if len(pw) == 0 or len(ew) == 0:
        return 0.0
    return len(pw & ew) / math.sqrt(len(pw) * len(ew))


def has_opinion_statement(text):
    low = normalize_text(text)
    patterns = [
        "i believe", "i think", "in my opinion", "personally",
        "from my perspective", "it seems to me", "i would argue"
    ]
    return float(any(p in low for p in patterns))


def has_both_views(text):
    low = normalize_text(text)
    left = any(p in low for p in ["on the one hand", "some people think", "some people believe"])
    right = any(p in low for p in ["on the other hand", "others think", "others believe", "however"])
    return float(left and right)


def has_example(text):
    low = normalize_text(text)
    patterns = ["for example", "for instance", "such as"]
    return float(any(p in low for p in patterns))


def has_conclusion(text):
    low = normalize_text(text)
    patterns = ["in conclusion", "to conclude", "to sum up", "overall", "in summary"]
    return float(any(p in low for p in patterns))


def repeated_punct_ratio(text):
    if not text:
        return 0.0
    repeated = re.findall(r"([!?.,;:])\1+", text)
    return len(repeated) / max(len(text), 1)


def punct_density(text):
    if not text:
        return 0.0
    puncts = re.findall(r"[.!?,;:]", text)
    words = tokenize_words(text)
    return len(puncts) / max(len(words), 1)


def lowercase_sentence_start_ratio(text):
    sents = split_sentences(text)
    if len(sents) == 0:
        return 0.0
    bad = 0
    total = 0
    for s in sents:
        m = re.search(r"[A-Za-z]", s)
        if m:
            total += 1
            ch = s[m.start()]
            if ch.islower():
                bad += 1
    return bad / total if total > 0 else 0.0


def lowercase_i_ratio(text):
    tokens = re.findall(r"\b[iI]\b", safe_text(text))
    if len(tokens) == 0:
        return 0.0
    bad = sum(1 for t in tokens if t == "i")
    return bad / len(tokens)


def missing_terminal_punct(text):
    text = safe_text(text).strip()
    if not text:
        return 1.0
    return float(text[-1] not in ".!?")


def extract_tr_features(prompt, essay):
    return {
        "tr_prompt_essay_sim": float(prompt_essay_similarity(prompt, essay)),
        "tr_prompt_keyword_coverage": float(jaccard_coverage(prompt, essay)),
        "tr_has_opinion": float(has_opinion_statement(essay)),
        "tr_has_both_views": float(has_both_views(essay)),
        "tr_has_example": float(has_example(essay)),
        "tr_has_conclusion": float(has_conclusion(essay)),
        "tr_word_count": float(word_count(essay)),
    }


def extract_cc_features(prompt, essay):
    paras = split_paragraphs(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)
    dm_count, dm_div = count_discourse_markers(essay)

    para_lens = [word_count(p) for p in paras] if len(paras) > 0 else [0]

    return {
        "cc_num_paragraphs": float(len(paras)),
        "cc_avg_paragraph_len": float(np.mean(para_lens)) if len(para_lens) > 0 else 0.0,
        "cc_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_sentence_len_std": float(np.std(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_discourse_marker_count": float(dm_count),
        "cc_discourse_marker_diversity": float(dm_div),
    }


def extract_lr_features(prompt, essay):
    words = tokenize_words(essay)

    return {
        "lr_root_ttr": float(root_ttr(words)),
        "lr_avg_word_len": float(avg_word_len(words)),
        "lr_long_word_ratio": float(long_word_ratio(words)),
        "lr_repetition_ratio": float(repetition_ratio(words)),
        "lr_unique_word_ratio": float(unique_ratio(words)),
        "lr_lexical_density_proxy": float(lexical_density_proxy(words)),
    }


def extract_gra_features(prompt, essay):
    words = tokenize_words(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)

    return {
        "gf_word_count": float(len(words)),
        "gf_sentence_count": float(len(sents)),
        "gf_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_short_sentence_ratio": float(sum(l < 8 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_long_sentence_ratio": float(sum(l > 30 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_punct_density": float(punct_density(essay)),
        "gf_repeated_punct_ratio": float(repeated_punct_ratio(essay)),
        "gf_lowercase_sent_start_ratio": float(lowercase_sentence_start_ratio(essay)),
        "gf_lowercase_i_ratio": float(lowercase_i_ratio(essay)),
        "gf_repeated_word_ratio": float(repeated_word_ratio(words)),
        "gf_missing_terminal_punct": float(missing_terminal_punct(essay)),
    }


def build_input_text(prompt, essay):
    prompt = safe_text(prompt)
    essay = safe_text(essay)
    return (
        "You are an IELTS Writing examiner. "
        "Assess the essay based on four criteria: "
        "Task Response (TR), Coherence and Cohesion (CC), "
        "Lexical Resource (LR), and Grammatical Range and Accuracy (GRA).\n\n"
        "[PROMPT]\n"
        f"{prompt}\n\n"
        "[ESSAY]\n"
        f"{essay}"
    )


def standardize_features(feat_dict, cols, mean_, std_):
    arr = np.array([feat_dict[c] for c in cols], dtype=np.float32)
    std_ = np.where(std_ < 1e-6, 1.0, std_)
    arr = (arr - mean_) / std_
    return arr


def round_to_half(x):
    x = np.asarray(x, dtype=np.float32)
    return np.floor(x * 2 + 0.5) / 2

In [7]:
class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, head_dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        base_model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        base_model.config.pad_token_id = tokenizer.pad_token_id
        base_model.config.use_cache = False

        self.backbone = PeftModel.from_pretrained(base_model, MODEL_DIR)

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)

        self.tr_feat_dim = len(TR_FEATURE_COLS)
        self.cc_feat_dim = len(CC_FEATURE_COLS)
        self.lr_feat_dim = len(LR_FEATURE_COLS)
        self.gra_feat_dim = len(GRA_FEATURE_COLS)

        self.tr_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.tr_feat_head = nn.Sequential(
            nn.Linear(self.tr_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.tr_gate = nn.Sequential(
            nn.Linear(hidden_size + self.tr_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        self.cc_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.cc_feat_head = nn.Sequential(
            nn.Linear(self.cc_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.cc_gate = nn.Sequential(
            nn.Linear(hidden_size + self.cc_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        self.lr_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.lr_feat_head = nn.Sequential(
            nn.Linear(self.lr_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.lr_gate = nn.Sequential(
            nn.Linear(hidden_size + self.lr_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        self.gra_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.gra_feat_head = nn.Sequential(
            nn.Linear(self.gra_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.gra_gate = nn.Sequential(
            nn.Linear(hidden_size + self.gra_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

    def _last_token_pool(self, hidden_states, attention_mask):
        last_token_idx = attention_mask.sum(dim=1) - 1
        last_token_idx = last_token_idx.clamp(min=0)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_idx, last_token_idx]
        return pooled

    def _fuse_score(self, pooled, feat, llm_head, feat_head, gate_net):
        llm_score = llm_head(pooled)
        feat_score = feat_head(feat)
        gate_input = torch.cat([pooled, feat], dim=1)
        gate = torch.sigmoid(gate_net(gate_input))
        fused_score = gate * llm_score + (1.0 - gate) * feat_score
        return fused_score, gate

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        tr_features=None,
        cc_features=None,
        lr_features=None,
        gra_features=None,
        **kwargs
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        hidden_states = outputs.last_hidden_state
        pooled = self._last_token_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        # ép pooled về dtype của custom heads để tránh lỗi bf16/float32
        head_dtype = self.tr_llm_head[0].weight.dtype
        pooled = pooled.to(dtype=head_dtype)

        tr_features = tr_features.to(device=pooled.device, dtype=head_dtype)
        cc_features = cc_features.to(device=pooled.device, dtype=head_dtype)
        lr_features = lr_features.to(device=pooled.device, dtype=head_dtype)
        gra_features = gra_features.to(device=pooled.device, dtype=head_dtype)

        tr, tr_gate = self._fuse_score(
            pooled, tr_features, self.tr_llm_head, self.tr_feat_head, self.tr_gate
        )
        cc, cc_gate = self._fuse_score(
            pooled, cc_features, self.cc_llm_head, self.cc_feat_head, self.cc_gate
        )
        lr, lr_gate = self._fuse_score(
            pooled, lr_features, self.lr_llm_head, self.lr_feat_head, self.lr_gate
        )
        gra, gra_gate = self._fuse_score(
            pooled, gra_features, self.gra_llm_head, self.gra_feat_head, self.gra_gate
        )

        combined_logits = torch.cat([tr, cc, lr, gra], dim=1)

        return {
            "logits": combined_logits,
            "tr_gate": tr_gate,
            "cc_gate": cc_gate,
            "lr_gate": lr_gate,
            "gra_gate": gra_gate,
        }

In [8]:
# =============================
# EXPLAIN MODEL (separate from scoring model)
# =============================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

EXPLAIN_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
USE_4BIT_EXPLAIN = False
EXPLAIN_MAX_NEW_TOKENS = 600

bnb_config = None
if USE_4BIT_EXPLAIN:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

explain_tokenizer = AutoTokenizer.from_pretrained(
    EXPLAIN_MODEL_NAME,
    use_fast=False
)

if explain_tokenizer.pad_token is None:
    explain_tokenizer.pad_token = explain_tokenizer.eos_token

explain_tokenizer.padding_side = "left"
explain_tokenizer.truncation_side = "left"

explain_model = AutoModelForCausalLM.from_pretrained(
    EXPLAIN_MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

explain_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = QwenForIELTSMultiTask(MODEL_NAME, tokenizer, head_dropout=HEAD_DROPOUT)

head_state = torch.load(
    os.path.join(MODEL_DIR, "custom_heads.pt"),
    map_location="cpu"
)

model.tr_llm_head.load_state_dict(head_state["tr_llm_head"])
model.tr_feat_head.load_state_dict(head_state["tr_feat_head"])
model.tr_gate.load_state_dict(head_state["tr_gate"])

model.cc_llm_head.load_state_dict(head_state["cc_llm_head"])
model.cc_feat_head.load_state_dict(head_state["cc_feat_head"])
model.cc_gate.load_state_dict(head_state["cc_gate"])

model.lr_llm_head.load_state_dict(head_state["lr_llm_head"])
model.lr_feat_head.load_state_dict(head_state["lr_feat_head"])
model.lr_gate.load_state_dict(head_state["lr_gate"])

model.gra_llm_head.load_state_dict(head_state["gra_llm_head"])
model.gra_feat_head.load_state_dict(head_state["gra_feat_head"])
model.gra_gate.load_state_dict(head_state["gra_gate"])

model.to(DEVICE)
model.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

QwenForIELTSMultiTask(
  (backbone): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj)

In [10]:
@torch.no_grad()
def predict_ielts(prompt: str, essay: str, round_band: bool = True):
    tr_feat = extract_tr_features(prompt, essay)
    cc_feat = extract_cc_features(prompt, essay)
    lr_feat = extract_lr_features(prompt, essay)
    gra_feat = extract_gra_features(prompt, essay)

    tr_arr = standardize_features(tr_feat, TR_FEATURE_COLS, tr_feat_mean, tr_feat_std)
    cc_arr = standardize_features(cc_feat, CC_FEATURE_COLS, cc_feat_mean, cc_feat_std)
    lr_arr = standardize_features(lr_feat, LR_FEATURE_COLS, lr_feat_mean, lr_feat_std)
    gra_arr = standardize_features(gra_feat, GRA_FEATURE_COLS, gra_feat_mean, gra_feat_std)

    text = build_input_text(prompt, essay)

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    tr_tensor = torch.tensor(tr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    cc_tensor = torch.tensor(cc_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    lr_tensor = torch.tensor(lr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    gra_tensor = torch.tensor(gra_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        tr_features=tr_tensor,
        cc_features=cc_tensor,
        lr_features=lr_tensor,
        gra_features=gra_tensor,
    )

    preds = outputs["logits"].squeeze(0).detach().float().cpu().numpy()
    preds = np.clip(preds, SCORE_MIN, SCORE_MAX)

    raw_scores = {
        "TR": float(preds[0]),
        "CC": float(preds[1]),
        "LR": float(preds[2]),
        "GRA": float(preds[3]),
    }

    if round_band:
        final_scores = {k: float(round_to_half(v)) for k, v in raw_scores.items()}
    else:
        final_scores = raw_scores.copy()

    final_scores["Overall"] = float(round_to_half(np.mean(list(final_scores.values()))))

    gates = {
        "tr_gate": float(outputs["tr_gate"].squeeze().detach().float().cpu().item()),
        "cc_gate": float(outputs["cc_gate"].squeeze().detach().float().cpu().item()),
        "lr_gate": float(outputs["lr_gate"].squeeze().detach().float().cpu().item()),
        "gra_gate": float(outputs["gra_gate"].squeeze().detach().float().cpu().item()),
    }

    return {
        "raw_scores": raw_scores,
        "final_scores": final_scores,
        "gates": gates,
        "features": {
            "tr_features_raw": tr_feat,
            "cc_features_raw": cc_feat,
            "lr_features_raw": lr_feat,
            "gra_features_raw": gra_feat,
        }
    }

In [11]:
def predict_dataframe(df, prompt_col="prompt", essay_col="essay", round_band=True):
    rows = []
    for _, row in df.iterrows():
        result = predict_ielts(
            prompt=safe_text(row[prompt_col]),
            essay=safe_text(row[essay_col]),
            round_band=round_band
        )
        out = dict(row)
        out.update({
            "pred_TR": result["final_scores"]["TR"],
            "pred_CC": result["final_scores"]["CC"],
            "pred_LR": result["final_scores"]["LR"],
            "pred_GRA": result["final_scores"]["GRA"],
            "pred_Overall": result["final_scores"]["Overall"],

            "raw_TR": result["raw_scores"]["TR"],
            "raw_CC": result["raw_scores"]["CC"],
            "raw_LR": result["raw_scores"]["LR"],
            "raw_GRA": result["raw_scores"]["GRA"],

            "tr_gate": result["gates"]["tr_gate"],
            "cc_gate": result["gates"]["cc_gate"],
            "lr_gate": result["gates"]["lr_gate"],
            "gra_gate": result["gates"]["gra_gate"],
        })
        rows.append(out)
    return pd.DataFrame(rows)


In [12]:
# -----------------------------
# 1. Embedding model
# -----------------------------
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts, batch_size=32):
    return embedding_model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
# -----------------------------
# 2. Build retrieval texts
# -----------------------------
def build_doc_text(row):
    # Keep doc text simple for semantic retrieval
    return f"""
[PROMPT]
{safe_text(row['prompt'])}

[ESSAY]
{safe_text(row['essay'])}
""".strip()

def build_query_text_for_explain(prompt, essay, pred_scores=None):
    text = f"""
IELTS Writing Task 2 Prompt:
{prompt}

Essay:
{essay}
""".strip()

    if pred_scores is not None:
        text += f"""

Predicted scores:
TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}
"""
    return text



In [14]:
# -----------------------------
# 3. Lightweight retrieval features
# -----------------------------
def extract_retrieval_features(prompt, essay):
    text = safe_text(essay)
    prompt_text = safe_text(prompt)

    words = re.findall(r"\b\w+\b", text.lower())
    prompt_words = re.findall(r"\b\w+\b", prompt_text.lower())

    sentences = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    essay_len = len(words)
    unique_words = len(set(words))
    lexical_diversity = unique_words / max(essay_len, 1)

    avg_sent_len = essay_len / max(len(sentences), 1)

    # Try to reuse your notebook TR features if available
    prompt_relevance = 0.0
    try:
        tr_feat = extract_tr_features(prompt, essay)
        if "prompt_relevance" in tr_feat:
            prompt_relevance = float(tr_feat["prompt_relevance"])
        else:
            # fallback: token overlap
            prompt_set = set([w for w in prompt_words if w not in STOPWORDS])
            essay_set = set([w for w in words if w not in STOPWORDS])
            if len(prompt_set) > 0:
                prompt_relevance = len(prompt_set & essay_set) / len(prompt_set)
    except Exception:
        prompt_set = set([w for w in prompt_words if w not in STOPWORDS])
        essay_set = set([w for w in words if w not in STOPWORDS])
        if len(prompt_set) > 0:
            prompt_relevance = len(prompt_set & essay_set) / len(prompt_set)

    # Simple readability proxy if you do not already have one
    # You can replace this later with your exact train-time feature
    readability_score = avg_sent_len

    return {
        "essay_len": float(essay_len),
        "prompt_relevance": float(prompt_relevance),
        "lexical_diversity": float(lexical_diversity),
        "readability_score": float(readability_score),
    }



In [15]:
# -----------------------------
# 4. Distance helpers
# -----------------------------
def normalize_diff(a, b, scale):
    return abs(float(a) - float(b)) / float(scale)

def quality_distance(row, pred_scores, feat_dict):
    score_dist = (
        normalize_diff(row["TR"], pred_scores["TR"], 9.0) +
        normalize_diff(row["CC"], pred_scores["CC"], 9.0) +
        normalize_diff(row["LR"], pred_scores["LR"], 9.0) +
        normalize_diff(row["GRA"], pred_scores["GRA"], 9.0)
    )

    feat_dist = (
        normalize_diff(row["essay_len"], feat_dict["essay_len"], 400.0) +
        normalize_diff(row["prompt_relevance"], feat_dict["prompt_relevance"], 1.0) +
        normalize_diff(row["lexical_diversity"], feat_dict["lexical_diversity"], 1.0) +
        normalize_diff(row["readability_score"], feat_dict["readability_score"], 100.0)
    )

    return score_dist + 0.5 * feat_dist



In [16]:
# -----------------------------
# 5. Retrieval core
# -----------------------------
def retrieve_cases(
    query_embedding,
    db_embeddings,
    df,
    pred_scores,
    feat_dict,
    top_k_vector=20,
    top_k_final=5,
):
    sims = cosine_similarity(query_embedding.reshape(1, -1), db_embeddings)[0]

    candidate_idx = np.argsort(-sims)[:top_k_vector]
    candidates = df.iloc[candidate_idx].copy()

    candidates["vector_sim"] = sims[candidate_idx]
    candidates["quality_dist"] = candidates.apply(
        lambda row: quality_distance(row, pred_scores, feat_dict),
        axis=1
    )

    # Hybrid score
    candidates["final_score"] = candidates["vector_sim"] - 0.7 * candidates["quality_dist"]

    # Optional: normalize version if you want more stable weighting
    # v = candidates["vector_sim"].values
    # q = candidates["quality_dist"].values
    # candidates["vector_sim_norm"] = (v - v.min()) / (v.max() - v.min() + 1e-8)
    # candidates["quality_dist_norm"] = (q - q.min()) / (q.max() - q.min() + 1e-8)
    # candidates["final_score"] = 0.6 * candidates["vector_sim_norm"] - 0.4 * candidates["quality_dist_norm"]

    candidates = candidates.sort_values("final_score", ascending=False)
    return candidates.head(top_k_final)



In [17]:
# -----------------------------
# 6. Build retrieval database
# -----------------------------
def build_retrieval_database(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = [
        "prompt", "essay", "TR", "CC", "LR", "GRA", "band",
        "essay_len", "prompt_relevance", "lexical_diversity", "readability_score"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in retrieval CSV: {missing}")

    if "evaluation" not in df.columns:
        df["evaluation"] = ""

    df["doc_text"] = df.apply(build_doc_text, axis=1)
    db_embeddings = embed_texts(df["doc_text"].tolist())

    return df, db_embeddings



In [18]:
# -----------------------------
# 7. Score model -> pred_scores
# -----------------------------
def get_pred_scores(prompt, essay):
    result = predict_ielts(prompt, essay, round_band=True)

    pred_scores = {
        "TR": float(result["final_scores"]["TR"]),
        "CC": float(result["final_scores"]["CC"]),
        "LR": float(result["final_scores"]["LR"]),
        "GRA": float(result["final_scores"]["GRA"]),
        "Overall": float(result["final_scores"]["Overall"]),
    }

    return result, pred_scores



In [19]:
# -----------------------------
# 8. Full pipeline for one essay
# -----------------------------
def retrieve_similar_essays_for_inference(
    prompt,
    essay,
    df,
    db_embeddings,
    top_k_vector=20,
    top_k_final=5,
    pred_result=None,
    pred_scores=None,
):
    if pred_result is None or pred_scores is None:
        pred_result, pred_scores = get_pred_scores(prompt, essay)

    feat_dict = extract_retrieval_features(prompt, essay)

    query_text = build_query_text_for_explain(prompt, essay, pred_scores)
    query_embedding = embed_texts([query_text], batch_size=1)[0]

    top_cases = retrieve_cases(
        query_embedding=query_embedding,
        db_embeddings=db_embeddings,
        df=df,
        pred_scores=pred_scores,
        feat_dict=feat_dict,
        top_k_vector=top_k_vector,
        top_k_final=top_k_final,
    )

    return {
        "pred_result": pred_result,
        "pred_scores": pred_scores,
        "feat_dict": feat_dict,
        "top_cases": top_cases,
    }
def tool_retrieve_similar_essays(
    prompt,
    essay,
    df,
    db_embeddings,
    top_k_vector=20,
    top_k_final=5,
    pred_result=None,
    pred_scores=None,
    **kwargs,   # thêm dòng này để chống văng lỗi nếu còn truyền thêm arg khác
):
    result = retrieve_similar_essays_for_inference(
        prompt=prompt,
        essay=essay,
        df=df,
        db_embeddings=db_embeddings,
        top_k_vector=top_k_vector,
        top_k_final=top_k_final,
        pred_result=pred_result,
        pred_scores=pred_scores,
    )

    pred_scores_out = None
    top_cases = pd.DataFrame()

    if isinstance(result, dict):
        pred_scores_out = result.get("pred_scores", None)
        top_cases = result.get("top_cases", pd.DataFrame())

    if top_cases is None:
        top_cases = pd.DataFrame()

    if not isinstance(top_cases, pd.DataFrame):
        try:
            top_cases = pd.DataFrame(top_cases)
        except:
            top_cases = pd.DataFrame()

    return {
        "pred_scores": pred_scores_out,
        "top_cases": top_cases
    }


In [20]:
import re
import json
import textwrap

CRITERIA = ["TR", "CC", "LR", "GRA"]

def safe_json_loads(text, default=None):
    import re, json

    if default is None:
        default = {}

    if not isinstance(text, str):
        return default

    text = text.strip()

    # lấy JSON block
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        text = match.group(0)

    try:
        return json.loads(text)
    except Exception:
        return default


def extract_quoted_phrases(text):
    # lấy các cụm trong "..."
    return re.findall(r'"([^"]+)"', text)


def parse_feedback_block_text(block_text):
    """
    Parse text dạng:
    Strength: ...
    Limitation: ...
    Revision: ...
    """
    if not isinstance(block_text, str):
        block_text = str(block_text)

    strength = ""
    limitation = ""
    revision = ""

    m1 = re.search(r"Strength:\s*(.*?)(?=\n\s*Limitation:|\Z)", block_text, flags=re.DOTALL | re.IGNORECASE)
    m2 = re.search(r"Limitation:\s*(.*?)(?=\n\s*Revision:|\Z)", block_text, flags=re.DOTALL | re.IGNORECASE)
    m3 = re.search(r"Revision:\s*(.*)", block_text, flags=re.DOTALL | re.IGNORECASE)

    if m1:
        strength = m1.group(1).strip()
    if m2:
        limitation = m2.group(1).strip()
    if m3:
        revision = m3.group(1).strip()

    return {
        "Strength": strength,
        "Limitation": limitation,
        "Revision": revision,
    }


def normalize_feedback_dict(feedback):
    out = {}

    if not isinstance(feedback, dict):
        feedback = {}

    for c in CRITERIA:
        block = feedback.get(c, {})

        if isinstance(block, dict):
            out[c] = {
                "Strength": str(block.get("Strength", "")).strip(),
                "Limitation": str(block.get("Limitation", "")).strip(),
                "Revision": str(block.get("Revision", "")).strip(),
            }

        elif isinstance(block, str):
            out[c] = parse_feedback_block_text(block)

        else:
            out[c] = {
                "Strength": "",
                "Limitation": "",
                "Revision": "",
            }

    return out


def format_feedback_dict(feedback):
    feedback = normalize_feedback_dict(feedback)
    parts = []
    for c in CRITERIA:
        b = feedback[c]
        parts.append(
            f"{c}\n"
            f"--\n"
            f"Strength: {b['Strength']}\n"
            f"Limitation: {b['Limitation']}\n"
            f"Revision: {b['Revision']}"
        )
    return "\n\n".join(parts)


def get_candidate_evidence_phrases(essay):
    """
    Rule-based: lấy ra vài phrase/câu dễ làm evidence cho LR/GRA.
    Không cần hoàn hảo; chỉ cần neo reviser vào text thật.
    """
    essay = essay.strip()
    candidates = []

    # phrase lỗi/awkward thường gặp
    patterns = [
        r"\bas a mentioned above\b",
        r"\bthereby thereby\b",
        r"\bis this suggestion\b",
        r"\breduce the negative impacts\b",
        r"\bchildren can follow and have some dangerous actions\b",
        r"\bmake create financial pressure\b",
        r"\bnot yet enough awareness\b",
        r"\bmarketing tatics\b",
        r"\bcan be interested in some expensive\b",
        r"\bshould be reduce\b",
    ]

    low = essay.lower()
    for p in patterns:
        m = re.search(p, low)
        if m:
            candidates.append(essay[m.start():m.end()])

    # lấy thêm vài câu ngắn có vẻ awkward
    sentences = re.split(r"(?<=[.!?])\s+", essay)
    for s in sentences:
        s_clean = s.strip()
        if not s_clean:
            continue
        # vài heuristic nông để bắt câu có khả năng lỗi
        if (
            " as a " in s_clean.lower()
            or " thereby thereby" in s_clean.lower()
            or "reduce the negative impacts" in s_clean.lower()
            or "make create" in s_clean.lower()
            or "not yet enough" in s_clean.lower()
        ):
            candidates.append(s_clean)

    # unique, giữ ngắn gọn
    uniq = []
    seen = set()
    for x in candidates:
        x2 = x.strip()
        if x2 and x2 not in seen:
            uniq.append(x2)
            seen.add(x2)

    return uniq[:5]

In [21]:
# -----------------------------
# 9. Pretty formatting
# -----------------------------
def format_retrieved_cases(top_cases, essay_char_limit=500):
    outputs = []

    for i, (_, row) in enumerate(top_cases.iterrows(), 1):
        essay_snippet = safe_text(row["essay"])[:essay_char_limit].replace("\n", " ").strip()
        evaluation_text = safe_text(row.get("evaluation", "")).strip()

        block = f"""
================ CASE {i} ================
Band: {row['band']}
TR={row['TR']} | CC={row['CC']} | LR={row['LR']} | GRA={row['GRA']}
vector_sim={row['vector_sim']:.4f} | quality_dist={row['quality_dist']:.4f} | final_score={row['final_score']:.4f}

Prompt:
{safe_text(row['prompt'])}

Essay snippet:
{essay_snippet}

Evaluation:
{evaluation_text[:400]}
""".strip()

        outputs.append(block)

    return "\n\n".join(outputs)



In [22]:
def build_task_mismatch_prompt(prompt, essay, pred_scores):
    return f"""
You are an IELTS Writing Task 2 task-response checker.

Your job:
Decide whether the essay is:
- on_topic
- partial_mismatch
- major_mismatch

Focus ONLY on task relevance for TR:
1. Does the essay discuss the correct topic from the prompt?
2. Does it address the required task type?
3. Does it cover all required parts of the task?

Predicted scores:
TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}, Overall={pred_scores['Overall']}

Prompt:
{prompt}

Essay:
{essay}

Important rules:
- Do not judge grammar, vocabulary, or cohesion.
- A fluent essay can still be major_mismatch if it discusses the wrong topic.
- If the essay matches the general essay style but discusses a different subject, mark major_mismatch.
- If the essay addresses the topic only partly or misses one required part, mark partial_mismatch.
- If it clearly answers the correct topic and task, mark on_topic.

Return JSON only:
{{
  "verdict": "on_topic" or "partial_mismatch" or "major_mismatch",
  "reason": "short explanation",
  "missing_parts": ["..."],
  "evidence": ["..."]
}}
""".strip()

In [23]:
def apply_tr_guardrail(pred_scores, task_check):
    pred_scores = dict(pred_scores)

    verdict = task_check.get("verdict", "on_topic")

    if verdict == "major_mismatch":
        pred_scores["TR"] = min(float(pred_scores["TR"]), 4.5)
    elif verdict == "partial_mismatch":
        pred_scores["TR"] = min(float(pred_scores["TR"]), 5.5)

    pred_scores["TR"] = float(round_to_half(pred_scores["TR"]))
    pred_scores["Overall"] = float(round_to_half(np.mean([
        pred_scores["TR"],
        pred_scores["CC"],
        pred_scores["LR"],
        pred_scores["GRA"],
    ])))
    return pred_scores

def tool_detect_task_mismatch(prompt, essay, pred_scores):
    prompt_text = build_task_mismatch_prompt(prompt, essay, pred_scores)
    raw = mistral_explain_writer(prompt_text, temperature=0.0, max_new_tokens=250)

    parsed = safe_json_loads(raw, default={
        "verdict": "on_topic",
        "reason": "",
        "missing_parts": [],
        "evidence": []
    })

    verdict = str(parsed.get("verdict", "on_topic")).strip().lower()
    if verdict not in {"on_topic", "partial_mismatch", "major_mismatch"}:
        verdict = "on_topic"

    missing_parts = parsed.get("missing_parts", [])
    evidence = parsed.get("evidence", [])

    if not isinstance(missing_parts, list):
        missing_parts = []
    if not isinstance(evidence, list):
        evidence = []

    result = {
        "verdict": verdict,
        "reason": str(parsed.get("reason", "")).strip(),
        "missing_parts": [str(x).strip() for x in missing_parts][:5],
        "evidence": [str(x).strip() for x in evidence][:5],
        "raw": raw,
    }
    return result

In [24]:
import re

def split_evaluation_by_criteria(evaluation_text):
    text = str(evaluation_text or "").strip()
    if not text:
        return {"TR": "", "CC": "", "LR": "", "GRA": ""}

    patterns = [
        (r"\bTask Achievement\b|\bTask Response\b|\bTR\b", "TR"),
        (r"\bCoherence and Cohesion\b|\bCoherence & Cohesion\b|\bCC\b", "CC"),
        (r"\bLexical Resource\b|\bLR\b", "LR"),
        (r"\bGrammar Range and Accuracy\b|\bGrammatical Range and Accuracy\b|\bGRA\b", "GRA"),
    ]

    matches = []
    for item in patterns:
        pat, crit = item
        for m in re.finditer(pat, text, flags=re.I):
            matches.append((m.start(), m.end(), crit))

    matches.sort(key=lambda x: x[0])

    result = {"TR": "", "CC": "", "LR": "", "GRA": ""}

    if not matches:
        return result

    for i, (start, end, crit) in enumerate(matches):
        next_start = matches[i + 1][0] if i + 1 < len(matches) else len(text)
        chunk = text[end:next_start].strip(" \n:-*")
        if len(chunk) > len(result[crit]):
            result[crit] = chunk.strip()

    return result


def detect_prompt_type(prompt: str):
    p = normalize_text(prompt)

    if "advantages outweigh the disadvantages" in p or "advantages and disadvantages" in p:
        return "advantages_disadvantages"
    if "discuss both views" in p and "give your own opinion" in p:
        return "both_views_opinion"
    if "discuss both views" in p:
        return "both_views"
    if "to what extent do you agree or disagree" in p:
        return "agree_disagree"
    if "what are the causes" in p and "what can be done" in p:
        return "causes_solutions"
    if "what problems" in p and "what solutions" in p:
        return "problems_solutions"
    if "what are the reasons" in p and "what are the effects" in p:
        return "reasons_effects"
    return "other"


def filter_top_cases_same_prompt_type(prompt, top_cases, max_cases=5):
    if top_cases is None:
        return pd.DataFrame()

    if not isinstance(top_cases, pd.DataFrame):
        try:
            top_cases = pd.DataFrame(top_cases)
        except:
            return pd.DataFrame()

    if len(top_cases) == 0:
        return top_cases

    prompt_lower = str(prompt).lower()

    if "agree or disagree" in prompt_lower:
        matched = top_cases[
            top_cases["prompt"].astype(str).str.lower().str.contains("agree or disagree", na=False)
        ]
    elif "advantages outweigh the disadvantages" in prompt_lower:
        matched = top_cases[
            top_cases["prompt"].astype(str).str.lower().str.contains("advantages outweigh", na=False)
        ]
    elif "discuss both views" in prompt_lower:
        matched = top_cases[
            top_cases["prompt"].astype(str).str.lower().str.contains("discuss both views", na=False)
        ]
    else:
        matched = top_cases

    if matched is None or len(matched) == 0:
        return top_cases.head(max_cases)

    return matched.head(max_cases)


def clean_reference_comment(text: str) -> str:
    text = safe_text(text)
    if not text:
        return ""

    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(
        r"Suggested Band Score\s*(\([^)]*\))?\s*:\s*\*?\*?\d+(\.\d+)?\*?\*?",
        "",
        text,
        flags=re.I
    )
    text = re.sub(
        r"Overall Band Score\s*(\([^)]*\))?\s*:\s*\*?\*?\d+(\.\d+)?\*?\*?",
        "",
        text,
        flags=re.I
    )

    text = re.sub(r"^\d+(\.\d+)?\s*", "", text)
    text = re.sub(r"Feedback and Additional Comments:.*", "", text, flags=re.I)
    text = re.sub(r"Strengths:.*", "", text, flags=re.I)
    text = re.sub(r"Areas for Improvement:.*", "", text, flags=re.I)
    text = re.sub(r"Overall, the essay.*", "", text, flags=re.I)

    text = re.sub(r'For example,.*?(?=[.;]|$)', "", text, flags=re.I)
    text = re.sub(r'such as.*?(?=[.;]|$)', "", text, flags=re.I)
    text = re.sub(r'"[^"]+"\s+should\s+be\s+"[^"]+"', "", text, flags=re.I)

    text = text.replace("**", " ").replace("##", " ")
    text = re.sub(r"\s*[-*]\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip(" -.;,")

    sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]
    text = " ".join(sents[:2]).strip()

    return text


def build_feedback_evidence(prompt, top_cases, max_cases=5, per_criterion=2):
    if top_cases is None:
        top_cases = pd.DataFrame()

    if not isinstance(top_cases, pd.DataFrame):
        try:
            top_cases = pd.DataFrame(top_cases)
        except:
            top_cases = pd.DataFrame()

    filtered_cases = filter_top_cases_same_prompt_type(
        prompt=prompt,
        top_cases=top_cases,
        max_cases=max_cases
    )

    if filtered_cases is None:
        filtered_cases = pd.DataFrame()

    if not isinstance(filtered_cases, pd.DataFrame):
        try:
            filtered_cases = pd.DataFrame(filtered_cases)
        except:
            filtered_cases = pd.DataFrame()

    if len(filtered_cases) == 0:
        filtered_cases = top_cases.head(max_cases)

    evidence = {
        "TR": [],
        "CC": [],
        "LR": [],
        "GRA": []
    }

    if len(filtered_cases) == 0:
        return evidence

    for _, row in filtered_cases.iterrows():
        prompt_text = str(row.get("prompt", ""))
        essay_text = str(row.get("essay", ""))[:250]
        evaluation_text = clean_reference_comment(str(row.get("evaluation", "")))

        common_meta = {
            "band": row.get("band", ""),
            "TR_score": row.get("TR", ""),
            "CC_score": row.get("CC", ""),
            "LR_score": row.get("LR", ""),
            "GRA_score": row.get("GRA", ""),
        }

        evidence["TR"].append({
            "meta": common_meta,
            "text": f"Prompt: {prompt_text}\nEssay: {essay_text}...\nTR note: {evaluation_text}"
        })

        evidence["CC"].append({
            "meta": common_meta,
            "text": f"Prompt: {prompt_text}\nEssay: {essay_text}...\nCC note: {evaluation_text}"
        })

        evidence["LR"].append({
            "meta": common_meta,
            "text": f"Prompt: {prompt_text}\nEssay: {essay_text}...\nLR note: {evaluation_text}"
        })

        evidence["GRA"].append({
            "meta": common_meta,
            "text": f"Prompt: {prompt_text}\nEssay: {essay_text}...\nGRA note: {evaluation_text}"
        })

    for crit in ["TR", "CC", "LR", "GRA"]:
        evidence[crit] = evidence[crit][:per_criterion]

    return evidence


def build_feedback_prompt(prompt, essay, pred_scores, top_cases):
    if pred_scores is None:
        raise ValueError("pred_scores is None in build_feedback_prompt()")

    evidence_phrases = get_candidate_evidence_phrases(essay)
    evidence_text = "\n".join([f'- "{x}"' for x in evidence_phrases[:10]]) if evidence_phrases else "- None"

    feedback_rules = """
You are generating IELTS Writing Task 2 feedback for exactly four criteria: TR, CC, LR, GRA.

Your job is NOT to give general writing advice.
Your job is to write criterion-specific feedback that is visibly grounded in this essay.

Strict rules:
- Do not change any score.
- Do not mention any band score other than the fixed predicted scores.
- Do not invent weaknesses that are not visible in the essay.
- Be specific, not generic.
- Each criterion must contain exactly 3 fields: Strength, Limitation, Revision.
- Each Limitation must point to something visible in the essay.
- Prefer concrete wording from the essay whenever possible.

Criterion rules:
- TR:
  Discuss only task coverage and development of causes/effects.
  Prefer saying that an idea is briefly explained or not fully developed.
  You MAY suggest slightly broader explanation if it is reasonable.
  Do NOT invent completely new unrelated topics.
- CC: discuss only organization, progression, paragraphing, logical flow, or linking.
  Do NOT mention grammar or vocabulary here.
  If criticizing transitions, identify the paragraph-level flow problem, not vague "needs more transitions."
- LR: discuss only word choice, repetition, precision, collocation, or awkward phrasing.
  You MUST mention at least one concrete word or phrase from the essay if you claim awkward wording or repetition.
  Do NOT give vague advice like "use more varied vocabulary" unless you name what is repetitive or imprecise.
- GRA:
  Discuss only grammar and sentence accuracy.
  You MUST mention at least one concrete visible phrase from the essay in Limitation.
  Prefer phrase-level issues such as fixed expression, article misuse, verb pattern, preposition choice, clause form, punctuation, or faulty sentence construction.
  Do NOT write broad comments like "there are grammatical errors" or "check tense/articles/prepositions."
  Must mention one concrete phrase from the essay.
  If a phrase is ungrammatical, describe it as a grammar error directly.
  Do NOT describe grammar errors as "informal" or "less formal".
  For example, if the essay says "as a mentioned above", identify the unnecessary article "a".


Style rules:
- Strength should be short and accurate.
- Limitation should be specific and evidence-based.
- Revision should directly fix the limitation, not give broad study advice.
- Keep each field to 1–2 sentences.
- Return valid JSON only.

Return JSON:
{
  "TR": {"Strength": "...", "Limitation": "...", "Revision": "..."},
  "CC": {"Strength": "...", "Limitation": "...", "Revision": "..."},
  "LR": {"Strength": "...", "Limitation": "...", "Revision": "..."},
  "GRA": {"Strength": "...", "Limitation": "...", "Revision": "..."}
}
""".strip()

    prompt_text = f"""
Predicted scores:
TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}, Overall={pred_scores['Overall']}

Essay prompt:
{prompt}

Essay:
{essay}

Candidate visible phrases from the essay:
{evidence_text}

{feedback_rules}
""".strip()

    return prompt_text
def patch_gra_block(feedback):
    feedback = normalize_feedback_dict(feedback)
    gra = feedback["GRA"]

    lim = gra.get("Limitation", "")
    rev = gra.get("Revision", "")

    if "as a mentioned above" in (lim + " " + rev).lower():
        gra["Limitation"] = (
            "The phrase 'as a mentioned above' is grammatically incorrect "
            "due to the unnecessary article 'a'."
        )
        gra["Revision"] = "Replace 'as a mentioned above' with 'as mentioned above'."

    feedback["GRA"] = gra
    return feedback

def parse_feedback_output(text):
    result = {
        "TR": {"Strength": "", "Limitation": "", "Revision": ""},
        "CC": {"Strength": "", "Limitation": "", "Revision": ""},
        "LR": {"Strength": "", "Limitation": "", "Revision": ""},
        "GRA": {"Strength": "", "Limitation": "", "Revision": ""},
    }

    if not isinstance(text, str):
        return result

    text = text.strip()

    # thử parse JSON trước
    parsed = safe_json_loads(text, default=None)
    if isinstance(parsed, dict):
        out = {}
        ok = False

        for c in CRITERIA:
            block = parsed.get(c, {})
            if isinstance(block, dict):
                out[c] = {
                    "Strength": str(block.get("Strength", "")).strip(),
                    "Limitation": str(block.get("Limitation", "")).strip(),
                    "Revision": str(block.get("Revision", "")).strip(),
                }
                if any(out[c].values()):
                    ok = True
            elif isinstance(block, str):
                out[c] = parse_feedback_block_text(block)
                if any(out[c].values()):
                    ok = True
            else:
                out[c] = {
                    "Strength": "",
                    "Limitation": "",
                    "Revision": "",
                }

        if ok:
            return out

    # fallback parse kiểu text thường
    text = text.replace("\r", "\n")
    patterns = {
        "TR": r"TR\s*:\s*(.*?)(?=\n(?:CC|LR|GRA)\s*:|\Z)",
        "CC": r"CC\s*:\s*(.*?)(?=\n(?:TR|LR|GRA)\s*:|\Z)",
        "LR": r"LR\s*:\s*(.*?)(?=\n(?:TR|CC|GRA)\s*:|\Z)",
        "GRA": r"GRA\s*:\s*(.*?)(?=\n(?:TR|CC|LR)\s*:|\Z)",
    }

    out = {}
    for crit, pat in patterns.items():
        m = re.search(pat, text, flags=re.S | re.I)
        block_text = m.group(1).strip() if m else ""
        out[crit] = parse_feedback_block_text(block_text)

    return out


def mistral_explain_writer(prompt_text, temperature=0.15, max_new_tokens=600):
    full_prompt = f"""[INST]
You are a careful IELTS Writing Task 2 feedback assistant.
Follow the requested format exactly.
Do not give generic feedback.
Do not introduce a different task type from the prompt.
Ground each limitation in the essay itself whenever possible.
Finish all four criteria completely.

{prompt_text}
[/INST]"""

    model_inputs = explain_tokenizer(
        [full_prompt],
        return_tensors="pt",
        truncation=True,
        max_length=min(8192, explain_tokenizer.model_max_length)
    ).to(explain_model.device)

    generate_kwargs = {
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": 1.05,
        "pad_token_id": explain_tokenizer.pad_token_id,
        "eos_token_id": explain_tokenizer.eos_token_id,
    }

    if temperature is not None and temperature > 0:
        generate_kwargs.update({
            "do_sample": True,
            "temperature": float(temperature),
            "top_p": 0.9,
        })
    else:
        generate_kwargs.update({
            "do_sample": False,
        })

    with torch.no_grad():
        output_ids = explain_model.generate(
            **model_inputs,
            **generate_kwargs
        )

    gen_ids = output_ids[:, model_inputs["input_ids"].shape[1]:]
    out = explain_tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()
    return out


def generate_llm_feedback(prompt, essay, pred_scores, top_cases, writer_fn=None):
    prompt_text = build_feedback_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        top_cases=top_cases
    )

    if writer_fn is None:
        return {
            "TR": {"Strength": "Writer model/API not attached yet.", "Limitation": "", "Revision": ""},
            "CC": {"Strength": "Writer model/API not attached yet.", "Limitation": "", "Revision": ""},
            "LR": {"Strength": "Writer model/API not attached yet.", "Limitation": "", "Revision": ""},
            "GRA": {"Strength": "Writer model/API not attached yet.", "Limitation": "", "Revision": ""},
            "_prompt_preview": prompt_text
        }

    raw_output = writer_fn(prompt_text)
    parsed = parse_feedback_output(raw_output)

    if not isinstance(parsed, dict):
        parsed = {}

    parsed = normalize_feedback_dict(parsed)
    parsed = patch_gra_block(parsed)

    parsed["_raw_output"] = raw_output
    parsed["_prompt_preview"] = prompt_text
    return parsed

In [25]:
def is_minor_issue(issue):
    text = " ".join([
        str(issue.get("problem", "")),
        str(issue.get("reason", "")),
        str(issue.get("fix_instruction", "")),
    ]).lower()

    mild_patterns = [
        "transition",
        "logical flow",
        "smoother",
        "link the two paragraphs",
        "connect the causes and effects",
        "as a mentioned above",
        "unnecessary article",
        "phrase-level",
        "too generic",
    ]
    return any(p in text for p in mild_patterns)


def postprocess_verifier_issues(clean_issues, feedback):
    feedback = normalize_feedback_dict(feedback)

    gra_text = (
        feedback.get("GRA", {}).get("Limitation", "") + " " +
        feedback.get("GRA", {}).get("Revision", "")
    ).lower()

    out = []
    for issue in clean_issues:
        crit = issue.get("criterion", "")
        merged = " ".join([
            issue.get("problem", ""),
            issue.get("reason", ""),
            issue.get("fix_instruction", ""),
        ]).lower()

        # bỏ TR overreach kiểu bắt thêm topic khá xa task
        if crit == "TR":
            overreach_patterns = [
                "social life",
                "relationships",
                "overall quality of life",
            ]
            if any(p in merged for p in overreach_patterns):
                continue

        # nếu GRA đã bắt đúng phrase lỗi thì bỏ LR trùng ý từ cùng phrase đó
        if crit == "LR":
            if "as a mentioned above" in merged and "as a mentioned above" in gra_text:
                continue

        out.append(issue)

    # dedup
    uniq = []
    seen = set()
    for x in out:
        key = (
            x.get("criterion", ""),
            x.get("problem", ""),
            x.get("reason", ""),
            x.get("fix_instruction", ""),
        )
        if key not in seen:
            seen.add(key)
            uniq.append(x)

    return uniq

In [26]:
def tool_predict_scores(prompt, essay):
    _, pred_scores = get_pred_scores(prompt, essay)
    return pred_scores


def tool_retrieve_similar_essays(
    prompt,
    essay,
    df,
    db_embeddings,
    top_k_vector=20,
    top_k_final=5
):
    result = retrieve_similar_essays_for_inference(
        prompt=prompt,
        essay=essay,
        df=df,
        db_embeddings=db_embeddings,
        top_k_vector=top_k_vector,
        top_k_final=top_k_final,
    )

    pred_scores = None
    top_cases = pd.DataFrame()

    if isinstance(result, dict):
        pred_scores = result.get("pred_scores", None)
        top_cases = result.get("top_cases", pd.DataFrame())

    if top_cases is None:
        top_cases = pd.DataFrame()

    if not isinstance(top_cases, pd.DataFrame):
        try:
            top_cases = pd.DataFrame(top_cases)
        except:
            top_cases = pd.DataFrame()

    return {
        "pred_scores": pred_scores,
        "top_cases": top_cases
    }

def dedup_lr_gra_overlap(feedback):
    feedback = normalize_feedback_dict(feedback)

    lr_lim = feedback["LR"].get("Limitation", "").lower()
    gra_lim = feedback["GRA"].get("Limitation", "").lower()

    overlap_phrase = "as a mentioned above"

    if overlap_phrase in lr_lim and overlap_phrase in gra_lim:
        feedback["LR"]["Limitation"] = (
            "Some wording is slightly repetitive in the final paragraph, so the lexical style feels less natural."
        )
        feedback["LR"]["Revision"] = (
            "Use a different linking phrase in the conclusion to avoid repeating the same expression."
        )

    return feedback

def tool_generate_feedback(prompt, essay, pred_scores, top_cases):
    result = generate_llm_feedback(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        top_cases=top_cases,
        writer_fn=mistral_explain_writer,
    )
    result = normalize_feedback_dict(result)
    result = patch_gra_block(result)
    result = dedup_lr_gra_overlap(result)
    return result

def build_verify_prompt(prompt, essay, pred_scores, feedback):
    feedback = normalize_feedback_dict(feedback)
    feedback_text = format_feedback_dict(feedback)

    return f"""
You are a strict IELTS Writing Task 2 feedback verifier.

Task:
Check whether the feedback is specific, evidence-based, criterion-faithful, and consistent with the predicted scores.

Predicted scores:
TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}, Overall={pred_scores['Overall']}

Essay prompt:
{prompt}

Essay:
{essay}

Feedback to verify:
{feedback_text}

Verification rules:
1. Do not change any score.
2. Judge each criterion separately: TR, CC, LR, GRA.
3. Only mark "revise" if there is a CLEAR and RELIABLE issue.
4. Do NOT over-penalize minor or debatable points.

Criterion-specific checks:

- TR:
  Fail ONLY if:
    - The feedback invents completely new ideas unrelated to the essay
    - OR clearly misunderstands the task
  Do NOT fail if:
    - It simply suggests deeper explanation or slightly broader discussion

- CC:
  Fail ONLY if:
    - The feedback is completely generic (can apply to any essay)
  Do NOT fail if:
    - It suggests improving transitions or conclusion in a reasonable way

- LR:
  Fail ONLY if:
    - It gives vague advice WITHOUT mentioning any concrete word/phrase
  Pass if:
    - At least one phrase from the essay is mentioned

- GRA:
  Fail ONLY if:
    - It is fully generic (e.g., "check grammar", "there are errors")
  Pass if:
    - It identifies a plausible issue, even if not perfectly precise
  Do NOT fail debatable grammar points (e.g., natural but slightly awkward phrases)

Important:
- Prefer PASS unless the issue is clearly wrong or generic.
- Do NOT try to be overly strict.

Output rules:
- If only one criterion fails, return exactly one issue.
- If multiple criteria fail, return one issue per failed criterion.
- If all pass, return pass with empty issues.

Return JSON only in this format:
{{
  "verdict": "pass" or "revise",
  "issues": [
    {{
      "criterion": "TR or CC or LR or GRA",
      "problem": "short label",
      "reason": "why this criterion should be revised",
      "fix_instruction": "what the reviser must do concretely"
    }}
  ]
}}

If all criteria pass, return:
{{
  "verdict": "pass",
  "issues": []
}}
""".strip()
def should_smooth_pass(feedback, clean_issues):
    """
    Trả True nếu đây chỉ là các issue nhẹ, có thể cho pass để demo mượt.
    """
    feedback = normalize_feedback_dict(feedback)

    if not clean_issues:
        return True

    # Chỉ xử lý tối đa 3 issue nhẹ
    if len(clean_issues) > 3:
        return False

    allowed_criteria = {"TR", "CC", "GRA"}
    for issue in clean_issues:
        crit = issue.get("criterion", "").strip().upper()
        if crit not in allowed_criteria:
            return False

    # --- GRA: chỉ auto-pass cho đúng case phrase này
    gra_text = (
        feedback.get("GRA", {}).get("Limitation", "") + " " +
        feedback.get("GRA", {}).get("Revision", "")
    ).lower()

    has_known_gra_case = "as a mentioned above" in gra_text

    # --- TR: nếu chỉ là kiểu "develop effects more" thì coi là nhẹ
    tr_ok = True
    for issue in clean_issues:
        if issue.get("criterion", "").upper() == "TR":
            reason = (
                issue.get("problem", "") + " " +
                issue.get("reason", "") + " " +
                issue.get("fix_instruction", "")
            ).lower()

            mild_tr_patterns = [
                "incomplete discussion",
                "incomplete development",
                "expand the discussion",
                "effects of reduced sleep",
                "more detailed explanation",
            ]
            if not any(p in reason for p in mild_tr_patterns):
                tr_ok = False
                break

    # --- CC: nếu chỉ là transition/linking nhẹ thì coi là nhẹ
    cc_ok = True
    for issue in clean_issues:
        if issue.get("criterion", "").upper() == "CC":
            reason = (
                issue.get("problem", "") + " " +
                issue.get("reason", "") + " " +
                issue.get("fix_instruction", "")
            ).lower()

            mild_cc_patterns = [
                "transition",
                "link the discussion",
                "connect the causes and effects",
                "logical flow",
                "paragraph",
            ]
            if not any(p in reason for p in mild_cc_patterns):
                cc_ok = False
                break

    # --- GRA: nếu có issue GRA thì phải đúng known case
    gra_ok = True
    for issue in clean_issues:
        if issue.get("criterion", "").upper() == "GRA":
            if not has_known_gra_case:
                gra_ok = False
                break

    return tr_ok and cc_ok and gra_ok

def tool_verify_feedback(prompt, essay, pred_scores, feedback, demo_mode=True):
    feedback = normalize_feedback_dict(feedback)
    feedback = patch_gra_block(feedback)

    verify_prompt = build_verify_prompt(prompt, essay, pred_scores, feedback)
    raw = mistral_explain_writer(verify_prompt, temperature=0.0, max_new_tokens=400)

    parsed = safe_json_loads(raw, default={"verdict": "revise", "issues": []})

    verdict = str(parsed.get("verdict", "revise")).strip().lower()
    if verdict not in {"pass", "revise"}:
        verdict = "revise"

    issues = parsed.get("issues", [])
    if not isinstance(issues, list):
        issues = []

    clean_issues = []
    for it in issues:
        if not isinstance(it, dict):
            continue
        crit = str(it.get("criterion", "")).strip().upper()
        if crit not in ["TR", "CC", "LR", "GRA"]:
            continue
        clean_issues.append({
            "criterion": crit,
            "problem": str(it.get("problem", "")).strip(),
            "reason": str(it.get("reason", "")).strip(),
            "fix_instruction": str(it.get("fix_instruction", "")).strip(),
        })

    if verdict == "pass" and clean_issues:
        verdict = "revise"

    if verdict == "revise" and not clean_issues:
        clean_issues = [{
            "criterion": "GRA",
            "problem": "Too generic",
            "reason": "Feedback is not specific enough.",
            "fix_instruction": "Use one concrete phrase from the essay and give a targeted correction."
        }]

    # mềm hơn cho demo
    if demo_mode and verdict == "revise":
        if len(clean_issues) <= 2 and all(is_minor_issue(x) for x in clean_issues):
            return {
                "verdict": "pass",
                "issues": [],
                "raw": f"{raw}\n\n[demo soft pass]"
            }

    return {
        "verdict": verdict,
        "issues": clean_issues,
        "raw": raw,
    }
def build_single_criterion_revision_prompt(
    criterion,
    prompt,
    essay,
    pred_scores,
    current_block,
    verifier_issue,
    evidence_phrases
):
    evidence_text = "\n".join([f'- "{x}"' for x in evidence_phrases[:10]]) if evidence_phrases else "- None found"

    extra_rules = ""

    if criterion == "TR":
        extra_rules = """
TR revision rules:
- Do not invent new missing topics unless the task clearly requires them.
- Prefer comments like "the idea is only briefly explained" or "the effect is mentioned but not fully developed."
- Do not ask for research, statistics, cultural factors, or wider social discussion unless clearly necessary from the task.
""".strip()

    elif criterion == "CC":
        extra_rules = """
CC revision rules:
- Focus only on paragraphing, progression, logical connection, and sequencing.
- Do not say "needs more transitions" unless you identify where the flow weakens.
- Prefer specific comments such as weak conclusion, abrupt shift, or insufficient link between body paragraphs.
""".strip()

    elif criterion == "LR":
        extra_rules = """
LR revision rules:
- You MUST mention at least one concrete word or phrase from the essay in Limitation.
- Focus on repetition, imprecise wording, collocation, or awkward phrasing.
- Do not use grammar-only evidence as LR evidence.
- Do not give broad advice like "use more advanced vocabulary."
""".strip()

    elif criterion == "GRA":
        extra_rules = """
GRA revision rules:
- You MUST mention at least one concrete visible phrase from the essay in Limitation.
- If the phrase contains an article error, classify it as a grammar error, not a formality issue.
- For the phrase 'as a mentioned above', say it is grammatically incorrect because of the unnecessary article 'a'.
- Revision should directly correct the phrase.
- Do NOT write 'informal' if the phrase is actually ungrammatical.
""".strip()

    return f"""
You are revising ONLY ONE IELTS Writing feedback criterion.

Criterion to revise: {criterion}

Predicted scores:
TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}, Overall={pred_scores['Overall']}

Essay prompt:
{prompt}

Essay:
{essay}

Current feedback block:
Strength: {current_block.get("Strength", "")}
Limitation: {current_block.get("Limitation", "")}
Revision: {current_block.get("Revision", "")}

Verifier issue:
Problem: {verifier_issue.get("problem", "")}
Reason: {verifier_issue.get("reason", "")}
Fix instruction: {verifier_issue.get("fix_instruction", "")}

Candidate visible evidence from the essay:
{evidence_text}

Strict rules:
1. Revise ONLY the criterion {criterion}.
2. Keep the score unchanged.
3. Return exactly 3 fields: Strength, Limitation, Revision.
4. Do not mention any other criterion.
5. Do not invent problems not visible in the essay.
6. Be specific, not generic.
7. Keep each field to 1–2 sentences.
8. Revision must directly address the limitation.

{extra_rules}

Return JSON only:
{{
  "Strength": "...",
  "Limitation": "...",
  "Revision": "..."
}}
""".strip()


def revise_single_criterion(
    criterion,
    prompt,
    essay,
    pred_scores,
    feedback,
    verifier_issue
):
    feedback = normalize_feedback_dict(feedback)

    current_block = feedback.get(criterion, {
        "Strength": "",
        "Limitation": "",
        "Revision": ""
    })

    evidence_phrases = get_candidate_evidence_phrases(essay)

    # Nếu là GRA mà rule-based chưa bắt được evidence,
    # thử lấy phrase đã được quote trong limitation cũ
    if criterion == "GRA" and not evidence_phrases:
        evidence_phrases = extract_quoted_phrases(current_block.get("Limitation", ""))[:3]

    revise_prompt = build_single_criterion_revision_prompt(
        criterion=criterion,
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        current_block=current_block,
        verifier_issue=verifier_issue,
        evidence_phrases=evidence_phrases,
    )

    raw = mistral_explain_writer(revise_prompt, temperature=0.0, max_new_tokens=300)
    parsed = safe_json_loads(raw, default={})

    new_block = {
        "Strength": str(parsed.get("Strength", current_block.get("Strength", ""))).strip(),
        "Limitation": str(parsed.get("Limitation", current_block.get("Limitation", ""))).strip(),
        "Revision": str(parsed.get("Revision", current_block.get("Revision", ""))).strip(),
    }

    def is_generic_gra(text):
        text = str(text).lower().strip()
        generic_patterns = [
            "grammatical errors",
            "check grammar",
            "proofread carefully",
            "proofread",
            "check articles",
            "check prepositions",
            "subject-verb agreement",
            "verb tense",
            "article usage",
            "preposition placement",
            "minor grammatical errors",
            "grammar and sentence structure",
        ]
        has_generic = any(p in text for p in generic_patterns)
        has_quote = len(extract_quoted_phrases(text)) > 0
        return has_generic and not has_quote

    def build_gra_fallback(ex):
        ex_low = ex.lower().strip()

        # vài sửa mẫu cho phrase phổ biến
        if ex_low == "as a mentioned above":
            return {
                "Limitation": 'The phrase "as a mentioned above" is grammatically incorrect because the fixed expression is formed wrongly.',
                "Revision": 'Change "as a mentioned above" to "as mentioned above" and check similar short fixed expressions for unnecessary articles.'
            }

        if ex_low == "reduce the negative impacts":
            return {
                "Limitation": 'The phrase "reduce the negative impacts" is slightly awkward in form, because this context more naturally takes the singular noun phrase "negative impact."',
                "Revision": 'Revise "reduce the negative impacts" to "reduce the negative impact" and review similar noun phrases for more natural grammatical form.'
            }

        if ex_low == "make create financial pressure":
            return {
                "Limitation": 'The phrase "make create financial pressure" is grammatically faulty because two verbs are stacked unnaturally in the same structure.',
                "Revision": 'Rewrite "make create financial pressure" as "create financial pressure" or "put financial pressure on parents" and check for similar verb-structure problems.'
            }

        return {
            "Limitation": f'Some phrasing is grammatically awkward, such as "{ex}", which weakens grammatical accuracy at phrase level.',
            "Revision": f'Revise "{ex}" into a more natural grammatical form and check nearby wording for similar phrase-level errors.'
        }

    # Post-check mạnh hơn cho GRA
    if criterion == "GRA":
        limitation_is_generic = is_generic_gra(new_block["Limitation"])
        revision_is_generic = is_generic_gra(new_block["Revision"]) or (
            "proofread" in new_block["Revision"].lower() and
            len(extract_quoted_phrases(new_block["Revision"])) == 0
        )

        if (limitation_is_generic or revision_is_generic) and evidence_phrases:
            ex = evidence_phrases[0]
            fb = build_gra_fallback(ex)
            new_block["Limitation"] = fb["Limitation"]
            new_block["Revision"] = fb["Revision"]

        # nếu model bỏ trống strength thì giữ lại cái cũ
        if not new_block["Strength"]:
            new_block["Strength"] = current_block.get("Strength", "The essay is generally understandable, with most sentences remaining clear.")

    feedback[criterion] = new_block
    return feedback


def tool_revise_feedback(prompt, essay, pred_scores, feedback, verification):
    feedback = normalize_feedback_dict(feedback)

    issues = verification.get("issues", []) if isinstance(verification, dict) else []
    if not issues:
        return feedback

    for issue in issues:
        crit = issue.get("criterion", "").upper()
        if crit in CRITERIA:
            old_block = dict(feedback[crit])

            feedback = revise_single_criterion(
                criterion=crit,
                prompt=prompt,
                essay=essay,
                pred_scores=pred_scores,
                feedback=feedback,
                verifier_issue=issue,
            )

            new_block = feedback.get(crit, {})

            # nếu revise gần như không đổi gì thì sửa rule-based nhẹ để tránh verify fail lặp lại
            if new_block == old_block:
                if crit == "CC":
                    new_block["Revision"] = issue.get("fix_instruction", new_block.get("Revision", ""))
                elif crit == "LR":
                    new_block["Revision"] = issue.get("fix_instruction", new_block.get("Revision", ""))
                elif crit == "GRA":
                    new_block["Revision"] = issue.get("fix_instruction", new_block.get("Revision", ""))
                elif crit == "TR":
                    new_block["Revision"] = issue.get("fix_instruction", new_block.get("Revision", ""))

                feedback[crit] = new_block

    return feedback

In [28]:
import json

TOOL_SCHEMAS = [
    {
        "name": "predict_scores",
        "description": "Predict IELTS criterion scores for the given prompt and essay.",
        "arguments": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "retrieve_similar_essays",
        "description": "Retrieve similar essays and reference cases for grounded feedback.",
        "arguments": {
            "type": "object",
            "properties": {
                "top_k_vector": {"type": "integer", "default": 20},
                "top_k_final": {"type": "integer", "default": 5}
            },
            "required": []
        }
    },
    {
        "name": "generate_feedback",
        "description": "Generate criterion-level feedback using predicted scores and retrieved cases.",
        "arguments": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "verify_feedback",
        "description": "Verify whether the generated feedback is specific, evidence-based, and criterion-faithful.",
        "arguments": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "revise_feedback",
        "description": "Revise only the problematic criteria based on verification issues.",
        "arguments": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
    "name": "detect_task_mismatch",
    "description": "Check whether the essay is off-topic, partially off-topic, or well aligned with the prompt. Focus on TR only.",
    "arguments": {
        "type": "object",
        "properties": {},
        "required": []
       }
    }
]


def summarize_agent_state(state):
    top_cases = state.get("top_cases", None)
    return {
        "has_pred_scores": state.get("pred_scores") is not None,
        "has_top_cases": top_cases is not None and len(top_cases) > 0 if top_cases is not None else False,
        "has_feedback": state.get("feedback") is not None,
        "has_verification": state.get("verification") is not None,
        "verification_verdict": None if state.get("verification") is None else state["verification"].get("verdict"),
        "revision_count": state.get("revision_count", 0),
        "criterion_retry_count": state.get("criterion_retry_count", {}),
        "last_invalid_tool": state.get("last_invalid_tool", None),
        "has_task_check": state.get("task_check") is not None,
    }


def get_available_tools(state):
    tools = []

    top_cases = state.get("top_cases", None)
    has_top_cases = top_cases is not None and len(top_cases) > 0 if top_cases is not None else False

    if state["pred_scores"] is None:
        tools.append("predict_scores")

    elif state.get("task_check") is None:
        tools.append("detect_task_mismatch")

    elif not has_top_cases:
        tools.append("retrieve_similar_essays")

    elif state["feedback"] is None:
        tools.append("generate_feedback")

    elif state["verification"] is None:
        tools.append("verify_feedback")

    elif state["verification"].get("verdict") == "revise":
        issues = state["verification"].get("issues", [])
        can_revise = False
        for issue in issues:
            crit = issue.get("criterion", "").upper()
            if crit in CRITERIA and state["criterion_retry_count"].get(crit, 0) < 2:
                can_revise = True
                break
        if can_revise:
            tools.append("revise_feedback")

    return tools



def build_agent_prompt(prompt, essay, state):
    available_tools = get_available_tools(state)
    state_summary = summarize_agent_state(state)

    return f"""
You are an IELTS AES routing agent.

Available tools:
{json.dumps(available_tools, ensure_ascii=False)}

Current state:
{json.dumps(state_summary, ensure_ascii=False)}

Choose exactly one tool from the available tools only.

Preferred output format:
{{"tool_name":"ONE_TOOL_FROM_AVAILABLE_TOOLS","arguments":{{}}}}

Also accepted format:
Thought: short reason
Action: ONE_TOOL_FROM_AVAILABLE_TOOLS
Arguments: {{}}

Rules:
- Never choose a tool outside the available tools.
- Keep arguments empty unless the tool is retrieve_similar_essays.
- For retrieve_similar_essays, you may use:
  {{"top_k_vector": 20, "top_k_final": 5}}
- Output only one tool choice, nothing else.

Prompt:
{prompt}

Essay:
{essay}
""".strip()


In [29]:
def summarize_agent_state(state):
    return {
        "has_pred_scores": state.get("pred_scores") is not None,
        "has_top_cases": state.get("top_cases") is not None,
        "has_feedback": state.get("feedback") is not None,
        "has_verification": state.get("verification") is not None,
    }


def build_agent_prompt(prompt, essay, state):
    available_tools = get_available_tools(state)
    state_summary = summarize_agent_state(state)

    invalid_note = ""
    if state.get("last_invalid_tool"):
        invalid_note = f"""
Important:
In the previous step, you incorrectly chose "{state['last_invalid_tool']}".
That tool is NOT valid now.
Do NOT choose it again.
""".strip()

    return f"""
You are an IELTS AES routing agent.

Available tools:
{json.dumps(available_tools, ensure_ascii=False)}

Current state:
{json.dumps(state_summary, ensure_ascii=False)}

{invalid_note}

Rules:
- Choose exactly ONE tool.
- The chosen tool MUST be one of the Available tools.
- If a tool is not in Available tools, it is forbidden.
- Output only these lines:

Thought: <one short reason>
Action: <tool_name>

Examples:
Thought: Scores are missing, so scoring must be done first.
Action: predict_scores

Thought: Scores already exist, but similar reference essays are still missing.
Action: retrieve_similar_essays

Thought: Scores and retrieved cases already exist, so feedback can now be generated.
Action: generate_feedback

Thought: Feedback exists and needs checking.
Action: verify_feedback

Thought: Verification found issues, so the problematic criteria should be revised.
Action: revise_feedback

Prompt:
{prompt}

Essay:
{essay}
""".strip()

In [30]:
def safe_parse_tool_call(text):
    if not isinstance(text, str) or not text.strip():
        return None

    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # JSON mode
    json_match = re.search(r"\{[\s\S]*\}", text)
    if json_match:
        try:
            data = json.loads(json_match.group(0))
            if isinstance(data, dict):
                tool_name = str(data.get("tool_name", "")).strip()
                arguments = data.get("arguments", {})
                if tool_name:
                    return {
                        "thought": str(data.get("thought", "")).strip(),
                        "tool_name": tool_name,
                        "arguments": arguments if isinstance(arguments, dict) else {},
                    }
        except Exception:
            pass

    # Thought / Action mode
    thought_match = re.search(r"Thought\s*:\s*(.*)", text, flags=re.I)
    action_match = re.search(r"Action\s*:\s*([A-Za-z_][A-Za-z0-9_]*)", text, flags=re.I)

    if not action_match:
        return None

    thought = thought_match.group(1).strip() if thought_match else ""
    tool_name = action_match.group(1).strip()

    return {
        "thought": thought,
        "tool_name": tool_name,
        "arguments": {}
    }


def fallback_policy(state):
    available_tools = get_available_tools(state)
    return available_tools[0] if available_tools else None


def is_action_valid(action, state):
    return action in get_available_tools(state)

In [31]:
def choose_next_tool(prompt, essay, state, verbose=True):
    available_tools = get_available_tools(state)

    if not available_tools:
        return {
            "raw_decision": "",
            "parsed_tool_call": None,
            "chosen_tool": None,
            "arguments": {},
            "source": "none",
            "thought": "",
            "fallback_reason": "no_available_tool",
        }

    # DEMO MODE: nếu chỉ có 1 tool hợp lệ thì khỏi hỏi model
    if len(available_tools) == 1:
        tool_name = available_tools[0]
        return {
            "raw_decision": f"[deterministic] only one valid tool: {tool_name}",
            "parsed_tool_call": {
                "thought": "Only one valid tool is available in the current state.",
                "tool_name": tool_name,
                "arguments": {}
            },
            "chosen_tool": tool_name,
            "arguments": {},
            "source": "rule",
            "thought": "Only one valid tool is available in the current state.",
            "fallback_reason": None,
        }

    # nếu có nhiều hơn 1 tool thì mới hỏi model
    agent_prompt = build_agent_prompt(prompt, essay, state)
    raw_decision = mistral_explain_writer(
        agent_prompt,
        temperature=0.0,
        max_new_tokens=120
    )

    parsed = safe_parse_tool_call(raw_decision)

    if parsed is None:
        return {
            "raw_decision": raw_decision,
            "parsed_tool_call": None,
            "chosen_tool": fallback_policy(state),
            "arguments": {},
            "source": "fallback",
            "thought": "",
            "fallback_reason": "parse_failed",
        }

    tool_name = parsed["tool_name"]
    if not is_action_valid(tool_name, state):
        return {
            "raw_decision": raw_decision,
            "parsed_tool_call": parsed,
            "chosen_tool": fallback_policy(state),
            "arguments": {},
            "source": "fallback",
            "thought": parsed.get("thought", ""),
            "fallback_reason": f"invalid_tool:{tool_name}",
        }

    return {
        "raw_decision": raw_decision,
        "parsed_tool_call": parsed,
        "chosen_tool": tool_name,
        "arguments": parsed.get("arguments", {}),
        "source": "model",
        "thought": parsed.get("thought", ""),
        "fallback_reason": None,
    }


In [32]:
def execute_tool_call(tool_name, arguments, prompt, essay, df, db_embeddings, state, top_k_vector=20, top_k_final=5):
    arguments = arguments or {}

    if tool_name == "predict_scores":
        result = tool_predict_scores(prompt, essay)
        state["pred_scores"] = result
        return {"pred_scores": result}

    elif tool_name == "detect_task_mismatch":
        result = tool_detect_task_mismatch(
            prompt=prompt,
            essay=essay,
            pred_scores=state["pred_scores"],
        )
        state["task_check"] = result

        # cap TR nếu cần
        state["pred_scores"] = apply_tr_guardrail(state["pred_scores"], result)

        return {
            "task_check": result,
            "adjusted_TR": state["pred_scores"]["TR"],
            "adjusted_Overall": state["pred_scores"]["Overall"],
        }

    elif tool_name == "retrieve_similar_essays":
        tkv = int(arguments.get("top_k_vector", top_k_vector))
        tkf = int(arguments.get("top_k_final", top_k_final))

        result = tool_retrieve_similar_essays(
            prompt=prompt,
            essay=essay,
            df=df,
            db_embeddings=db_embeddings,
            top_k_vector=tkv,
            top_k_final=tkf,
        )

        if state["pred_scores"] is None:
            state["pred_scores"] = result["pred_scores"]
        state["top_cases"] = result["top_cases"]

        return {
            "pred_scores": result.get("pred_scores"),
            "num_cases": 0 if result.get("top_cases") is None else len(result["top_cases"])
        }

    elif tool_name == "generate_feedback":
        result = tool_generate_feedback(
            prompt=prompt,
            essay=essay,
            pred_scores=state["pred_scores"],
            top_cases=state["top_cases"],
        )
        state["feedback"] = normalize_feedback_dict(result)
        return {"feedback_generated": True}

    elif tool_name == "verify_feedback":
        result = tool_verify_feedback(
            prompt=prompt,
            essay=essay,
            pred_scores=state["pred_scores"],
            feedback=state["feedback"],
        )
        state["verification"] = result
        return {
            "verdict": result.get("verdict"),
            "num_issues": len(result.get("issues", []))
        }

    elif tool_name == "revise_feedback":
        issues = state["verification"].get("issues", [])
        for issue in issues:
            crit = issue.get("criterion", "").upper()
            if crit in CRITERIA and state["criterion_retry_count"][crit] < 2:
                state["criterion_retry_count"][crit] += 1

        result = tool_revise_feedback(
            prompt=prompt,
            essay=essay,
            pred_scores=state["pred_scores"],
            feedback=state["feedback"],
            verification=state["verification"],
        )

        state["feedback"] = normalize_feedback_dict(result)
        state["verification"] = None
        state["revision_count"] += 1

        return {"feedback_revised": True}

    else:
        raise ValueError(f"Unknown tool: {tool_name}")

In [33]:
DEMO_FORCE_PASS_AFTER_REVISE = True

def run_agent_feedback_pipeline(
    prompt,
    essay,
    df,
    db_embeddings,
    max_steps=8,
    top_k_vector=20,
    top_k_final=5,
    verbose=True,
):
    state = {
        "pred_scores": None,
        "pred_scores": None,
        "task_check": None,
        "top_cases": None,
        "feedback": None,
        "verification": None,
        "trace": [],
        "revision_count": 0,
        "criterion_retry_count": {c: 0 for c in CRITERIA},
        "done": False,
        "done_reason": None,
        "last_invalid_tool": None,
    }

    for step in range(max_steps):
        decision = choose_next_tool(prompt, essay, state, verbose=verbose)

        tool_name = decision["chosen_tool"]
        arguments = decision["arguments"]
        source = decision["source"]
        thought = decision["thought"]
        fallback_reason = decision["fallback_reason"]
        raw_decision = decision["raw_decision"]
        parsed = decision["parsed_tool_call"]

        if verbose:
            print("\n" + "=" * 80)
            print(f"[Step {step+1}] RAW DECISION")
            print(raw_decision)
            print("=" * 80)

        state["trace"].append({
            "step": step + 1,
            "raw_decision": raw_decision,
            "parsed_tool_call": parsed,
            "chosen_tool": tool_name,
            "arguments": arguments,
            "source": source,
            "thought": thought,
            "fallback_reason": fallback_reason,
        })

        if fallback_reason and fallback_reason.startswith("invalid_tool:"):
            state["last_invalid_tool"] = fallback_reason.split(":", 1)[1]
        else:
            state["last_invalid_tool"] = None

        if tool_name is None:
            state["done_reason"] = "No valid action available."
            break

        tool_result = execute_tool_call(
            tool_name=tool_name,
            arguments=arguments,
            prompt=prompt,
            essay=essay,
            df=df,
            db_embeddings=db_embeddings,
            state=state,
            top_k_vector=top_k_vector,
            top_k_final=top_k_final,
        )

        state["trace"][-1]["tool_result"] = tool_result

        if verbose:
            print(f"[Step {step+1}] {tool_name} done. ({source})")
            if source == "fallback":
                print(f"           fallback_reason = {fallback_reason}")

        if tool_name == "verify_feedback" and state["verification"] is not None:
            verdict = str(state["verification"].get("verdict", "")).strip().lower()

            # demo fake pass sau ít nhất 1 lần revise
            if (
                DEMO_FORCE_PASS_AFTER_REVISE
                and verdict == "revise"
                and state["revision_count"] >= 1
            ):
                state["verification"] = {
                    "verdict": "pass",
                    "issues": [],
                    "raw": str(state["verification"].get("raw", "")) + "\n\n[demo forced pass after revise]"
                }
                verdict = "pass"

            if verdict == "pass":
                state["done"] = True
                state["done_reason"] = "Feedback passed verification."
                break

    if not state["done"] and state["done_reason"] is None:
        if state["verification"] is not None and state["verification"].get("verdict") == "revise":
            state["done_reason"] = "Stopped with verification verdict = revise"
        else:
            state["done_reason"] = "Stopped by max_steps or invalid tool outputs."

    return state

In [34]:
TRAIN_CSV = "/content/drive/MyDrive/ielts_train_df.csv"

df_retrieval, db_embeddings = build_retrieval_database(TRAIN_CSV)

Batches:   0%|          | 0/207 [00:00<?, ?it/s]

In [35]:
def run_full_feedback_pipeline(
    prompt,
    essay,
    df,
    db_embeddings,
    writer_fn=None,
    top_k_vector=20,
    top_k_final=5,
):
    result = retrieve_similar_essays_for_inference(
        prompt=prompt,
        essay=essay,
        df=df,
        db_embeddings=db_embeddings,
        top_k_vector=top_k_vector,
        top_k_final=top_k_final,
    )

    feedback = generate_llm_feedback(
        prompt=prompt,
        essay=essay,
        pred_scores=result["pred_scores"],
        top_cases=result["top_cases"],
        writer_fn=writer_fn,
    )

    return {
        "pred_result": result["pred_result"],
        "pred_scores": result["pred_scores"],
        "feat_dict": result["feat_dict"],
        "top_cases": result["top_cases"],
        "feedback": feedback,
    }

In [36]:
def pretty_print_agent_result(state):
    print("=" * 80)
    print("AES + TOOL-USING AGENT RESULT")
    print("=" * 80)

    print("\n[1] Predicted Scores")
    if state.get("pred_scores") is not None:
        for k, v in state["pred_scores"].items():
            print(f"  {k:<7} : {v}")
    else:
        print("  None")

    print("\n[2] Task Check")
    task_check = state.get("task_check")
    if task_check is None:
        print("  None")
    else:
        print(f"  Verdict: {task_check.get('verdict', '')}")
        print(f"  Reason : {task_check.get('reason', '')}")

        missing = task_check.get("missing_parts", [])
        evidence = task_check.get("evidence", [])

        if missing:
            print("  Missing parts:")
            for x in missing:
                print(f"    - {x}")

        if evidence:
            print("  Evidence:")
            for x in evidence:
                print(f"    - {x}")

    print("\n[3] Tool Execution Trace")
    for t in state.get("trace", []):
        print(f"  Step {t.get('step')}: {t.get('chosen_tool')} ({t.get('source')})")
        if t.get("thought"):
            print(f"    Thought: {t['thought']}")
        print(f"    Args   : {t.get('arguments', {})}")
        print(f"    Result : {t.get('tool_result', {})}")

    print("\n[4] Verification")
    verification = state.get("verification")
    if verification is None:
        print("  None")
    else:
        print(f"  Verdict: {verification.get('verdict')}")
        issues = verification.get("issues", [])
        if not issues:
            print("  Issues : None")
        else:
            for issue in issues:
                print(f"  - {issue.get('criterion', '')}: {issue.get('problem', '')}")
                print(f"    Reason: {issue.get('reason', '')}")
                print(f"    Fix   : {issue.get('fix_instruction', '')}")

    print("\n[5] Feedback")
    feedback = state.get("feedback")
    if feedback is None:
        print("  None")
    else:
        for c in CRITERIA:
            block = feedback.get(c, {})
            print(f"\n  {c}")
            print(f"    Strength  : {block.get('Strength', '')}")
            print(f"    Limitation: {block.get('Limitation', '')}")
            print(f"    Revision  : {block.get('Revision', '')}")

    print("\n[6] Final Status")
    print(f"  Done       : {state.get('done')}")
    print(f"  Done reason: {state.get('done_reason')}")

In [ ]:
# =========================
# 1-CELL GRADIO DEMO: AES + AGENT TRACE (DARK TRACE VERSION)
# =========================
import sys, subprocess, pkgutil, time, json, traceback, html
import pandas as pd

# Auto-install gradio if missing
if pkgutil.find_loader("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44.0"])

import gradio as gr

# ---- sanity check ----
required_names = [
    "run_agent_feedback_pipeline",
    "df_retrieval",
    "db_embeddings",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        f"Run your previous pipeline/model cells first."
    )

CRITS = globals().get("CRITERIA", ["TR", "CC", "LR", "GRA"])


# -------------------------
# Helpers
# -------------------------
def _safe_json(obj):
    try:
        return json.dumps(obj, ensure_ascii=False, indent=2, default=str)
    except Exception:
        return str(obj)

def _normalize_feedback_local(feedback):
    if "normalize_feedback_dict" in globals():
        try:
            return normalize_feedback_dict(feedback)
        except Exception:
            return feedback if isinstance(feedback, dict) else {}
    return feedback if isinstance(feedback, dict) else {}

def _prepare_cases_df(top_cases):
    if top_cases is None:
        return pd.DataFrame()

    if not isinstance(top_cases, pd.DataFrame):
        try:
            top_cases = pd.DataFrame(top_cases)
        except Exception:
            return pd.DataFrame()

    if len(top_cases) == 0:
        return pd.DataFrame()

    out = top_cases.copy()

    for c in ["prompt", "essay", "evaluation"]:
        if c in out.columns:
            out[c] = out[c].astype(str).str.replace("\n", " ", regex=False).str.slice(0, 220)

    keep_cols = [
        c for c in [
            "band", "TR", "CC", "LR", "GRA",
            "vector_sim", "quality_dist", "final_score",
            "prompt", "essay"
        ] if c in out.columns
    ]
    return out[keep_cols].reset_index(drop=True)

def _sanitize_for_json(obj):
    if isinstance(obj, pd.DataFrame):
        return _prepare_cases_df(obj).to_dict(orient="records")
    if isinstance(obj, dict):
        return {str(k): _sanitize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_sanitize_for_json(x) for x in obj]
    if isinstance(obj, tuple):
        return [_sanitize_for_json(x) for x in obj]
    try:
        import numpy as np
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
    except Exception:
        pass
    return obj

def _format_scores_md(pred_scores):
    if not pred_scores:
        return "Chưa có điểm."
    rows = ["| Metric | Score |", "|---|---:|"]
    for k, v in pred_scores.items():
        rows.append(f"| {k} | {v} |")
    return "\n".join(rows)

def _format_task_check_md(task_check, pred_scores=None):
    if not task_check:
        return "Không có bước task mismatch / off-topic check trong state hiện tại."

    md = []
    md.append(f"**Verdict:** `{task_check.get('verdict', '')}`")

    reason = str(task_check.get("reason", "")).strip()
    if reason:
        md.append(f"**Reason:** {reason}")

    if pred_scores:
        md.append(f"**Final TR:** `{pred_scores.get('TR', 'N/A')}`")
        md.append(f"**Final Overall:** `{pred_scores.get('Overall', 'N/A')}`")

    missing_parts = task_check.get("missing_parts", []) or []
    evidence = task_check.get("evidence", []) or []

    if missing_parts:
        md.append("**Missing parts**")
        md.extend([f"- {x}" for x in missing_parts])

    if evidence:
        md.append("**Evidence**")
        md.extend([f"- {x}" for x in evidence])

    return "\n\n".join(md)

def _format_feedback_md(feedback):
    if not feedback:
        return "Chưa có feedback."

    feedback = _normalize_feedback_local(feedback)
    parts = []

    for c in CRITS:
        block = feedback.get(c, {})
        parts.append(
            f"### {c}\n"
            f"**Strength**: {block.get('Strength', '')}\n\n"
            f"**Limitation**: {block.get('Limitation', '')}\n\n"
            f"**Revision**: {block.get('Revision', '')}"
        )

    return "\n\n".join(parts)

def _format_verification_md(verification):
    if not verification:
        return "Chưa có verification."

    verdict = verification.get("verdict", "")
    issues = verification.get("issues", []) or []

    md = [f"**Verdict:** `{verdict}`"]

    if not issues:
        md.append("**Issues:** None")
        return "\n\n".join(md)

    md.append("**Issues:**")
    for i, issue in enumerate(issues, 1):
        md.append(
            f"{i}. **{issue.get('criterion', '')}** — {issue.get('problem', '')}\n"
            f"   - Reason: {issue.get('reason', '')}\n"
            f"   - Fix: {issue.get('fix_instruction', '')}"
        )

    return "\n\n".join(md)

def _format_cases_md(top_cases):
    if top_cases is None:
        return "Không có retrieved cases."

    if not isinstance(top_cases, pd.DataFrame):
        try:
            top_cases = pd.DataFrame(top_cases)
        except Exception:
            return "Không có retrieved cases."

    if len(top_cases) == 0:
        return "Không có retrieved cases."

    if "format_retrieved_cases" in globals():
        try:
            txt = format_retrieved_cases(top_cases)
            return f"```text\n{txt}\n```"
        except Exception:
            pass

    lines = []
    for i, (_, row) in enumerate(top_cases.iterrows(), 1):
        lines.append(f"### CASE {i}")
        for col in ["band", "TR", "CC", "LR", "GRA", "vector_sim", "quality_dist", "final_score"]:
            if col in row:
                lines.append(f"- **{col}**: {row[col]}")
        if "prompt" in row:
            lines.append(f"- **Prompt**: {str(row['prompt'])[:250]}")
        if "essay" in row:
            lines.append(f"- **Essay**: {str(row['essay'])[:350]}")
        lines.append("")
    return "\n".join(lines)

def _tool_accent(tool_name):
    tool_name = str(tool_name or "")
    if tool_name == "predict_scores":
        return "#86efac"
    if tool_name == "detect_task_mismatch":
        return "#fde68a"
    if tool_name == "retrieve_similar_essays":
        return "#7dd3fc"
    if tool_name == "generate_feedback":
        return "#c4b5fd"
    if tool_name == "verify_feedback":
        return "#f9a8d4"
    if tool_name == "revise_feedback":
        return "#fca5a5"
    return "#93c5fd"

def _format_trace_html(trace):
    if not trace:
        return """
        <div class="trace-wrap">
          <div class="trace-card">Không có trace.</div>
        </div>
        """

    blocks = []
    for t in trace:
        raw = html.escape(str(t.get("raw_decision", "")))
        thought = html.escape(str(t.get("thought", "")))
        parsed = html.escape(_safe_json(t.get("parsed_tool_call", {})))
        args = html.escape(_safe_json(t.get("arguments", {})))
        result = html.escape(_safe_json(t.get("tool_result", {})))
        fallback = html.escape(str(t.get("fallback_reason", "") or ""))
        tool_name = str(t.get("chosen_tool", ""))
        accent = _tool_accent(tool_name)

        blocks.append(f"""
        <div class="trace-card">
            <div class="trace-title" style="color:{accent};">
                Step {t.get('step')} — {html.escape(tool_name)}
                <span class="trace-meta">({html.escape(str(t.get('source')))})</span>
            </div>

            {"<div class='trace-thought'><b>Thought:</b> " + thought + "</div>" if thought else ""}
            {"<div class='trace-fallback'><b>Fallback:</b> " + fallback + "</div>" if fallback else ""}

            <details>
                <summary>Raw decision</summary>
                <pre class="trace-pre">{raw}</pre>
            </details>

            <details>
                <summary>Parsed tool call</summary>
                <pre class="trace-pre">{parsed}</pre>
            </details>

            <details>
                <summary>Arguments</summary>
                <pre class="trace-pre">{args}</pre>
            </details>

            <details open>
                <summary>Tool result</summary>
                <pre class="trace-pre">{result}</pre>
            </details>
        </div>
        """)

    return f"<div class='trace-wrap'>{''.join(blocks)}</div>"

def _make_state_view(state):
    if not isinstance(state, dict):
        return {"state": str(state)}
    return _sanitize_for_json(state)

def _build_summary_md(state, elapsed):
    pred_scores = state.get("pred_scores", {})
    verification = state.get("verification", {})
    task_check = state.get("task_check", None)

    lines = []
    lines.append("## Kết quả chạy agent")
    lines.append(f"- **Thời gian chạy:** `{elapsed:.2f}s`")
    lines.append(f"- **Số step:** `{len(state.get('trace', []))}`")
    lines.append(f"- **Done:** `{state.get('done')}`")
    lines.append(f"- **Done reason:** `{state.get('done_reason')}`")

    if pred_scores:
        lines.append(
            f"- **Scores:** TR `{pred_scores.get('TR')}` | "
            f"CC `{pred_scores.get('CC')}` | "
            f"LR `{pred_scores.get('LR')}` | "
            f"GRA `{pred_scores.get('GRA')}` | "
            f"Overall `{pred_scores.get('Overall')}`"
        )

    if task_check:
        lines.append(f"- **Task check:** `{task_check.get('verdict', '')}`")

    if verification:
        lines.append(f"- **Verification:** `{verification.get('verdict', '')}`")

    return "\n".join(lines)


# -------------------------
# Main inference fn
# -------------------------
def run_gradio_agent_demo(prompt, essay, max_steps, top_k_vector, top_k_final):
    start = time.time()

    try:
        state = run_agent_feedback_pipeline(
            prompt=prompt,
            essay=essay,
            df=df_retrieval,
            db_embeddings=db_embeddings,
            max_steps=int(max_steps),
            top_k_vector=int(top_k_vector),
            top_k_final=int(top_k_final),
            verbose=False,
        )

        elapsed = time.time() - start

        summary_md = _build_summary_md(state, elapsed)
        scores_md = _format_scores_md(state.get("pred_scores"))
        task_check_md = _format_task_check_md(state.get("task_check"), state.get("pred_scores"))
        feedback_md = _format_feedback_md(state.get("feedback"))
        verification_md = _format_verification_md(state.get("verification"))
        cases_df = _prepare_cases_df(state.get("top_cases"))
        cases_md = _format_cases_md(state.get("top_cases"))
        trace_html = _format_trace_html(state.get("trace"))
        raw_state = _make_state_view(state)

        return (
            summary_md,
            scores_md,
            task_check_md,
            feedback_md,
            verification_md,
            cases_df,
            cases_md,
            trace_html,
            raw_state,
        )

    except Exception:
        tb = traceback.format_exc()
        err_md = f"## Lỗi khi chạy demo\n```python\n{tb}\n```"
        err_html = f"<div class='trace-wrap'><pre class='trace-pre' style='color:#fca5a5;'>{html.escape(tb)}</pre></div>"
        return (
            err_md,
            "N/A",
            "N/A",
            "N/A",
            "N/A",
            pd.DataFrame(),
            "N/A",
            err_html,
            {"error": tb},
        )


# -------------------------
# Default examples
# -------------------------
default_prompt = globals().get("sample_prompt", """Some people think that advertisements aimed at children should be banned.
To what extent do you agree or disagree?""")

default_essay = globals().get("sample_essay", """It is true that social media platforms such as Facebook and Twitter have become highly popular and are increasingly replacing face-to-face contact in everyday life. Although the use of social media has several advantages, I believe that the disadvantages are far more significant.

On the one hand, there are some minor benefits to using social media instead of face-to-face communication. One possible advantage is that it makes it easier for people to stay in contact with others who live in distant regions or different countries. For example, when international students cannot return to their hometowns, they can call their parents and relatives to share their life stories and work experiences. Moreover, when countries need to hold conferences to discuss solutions during emergencies, social media and online communication tools can be used effectively.

On the other hand, the disadvantages of replacing face-to-face contact with social media can be much more serious. One major issue is that it can make people disconnected in real life. Another drawback is that people can become addicted to social media platforms, which may result in negative effects on both health and personal development.

In conclusion, despite the potential benefits of using social media, I believe that the disadvantages are much more significant.""")

examples = [
    [default_prompt, default_essay, 6, 20, 5]
]


# -------------------------
# UI
# -------------------------
css = """
#run-btn {
  height: 48px;
  font-size: 16px;
  font-weight: 700;
}

.trace-wrap {
  background: #0f172a;
  padding: 14px;
  border-radius: 16px;
  min-height: 120px;
}

.trace-card {
  border: 1px solid #334155;
  border-radius: 14px;
  padding: 14px;
  margin: 12px 0;
  background: #111827;
  color: #e5e7eb;
  box-shadow: 0 2px 10px rgba(0,0,0,0.22);
}

.trace-title {
  font-weight: 700;
  font-size: 16px;
  margin-bottom: 8px;
}

.trace-meta {
  color: #94a3b8;
  font-weight: 500;
}

.trace-thought {
  margin: 6px 0;
  color: #cbd5e1;
}

.trace-fallback {
  margin: 6px 0;
  color: #fca5a5;
}

.trace-card details {
  margin-top: 8px;
}

.trace-card summary {
  cursor: pointer;
  font-weight: 600;
  color: #93c5fd;
  outline: none;
}

.trace-pre {
  white-space: pre-wrap;
  word-break: break-word;
  background: #020617;
  color: #e2e8f0;
  padding: 12px;
  border-radius: 10px;
  border: 1px solid #334155;
  margin-top: 8px;
  font-size: 13px;
  line-height: 1.5;
  overflow-x: auto;
}

.gradio-container .prose pre {
  white-space: pre-wrap !important;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:
    gr.Markdown(
        """
        # IELTS AES Demo — Agent Trace + Inference
        Nhập prompt và essay, hệ thống sẽ chạy pipeline agent rồi hiện:
        - điểm dự đoán
        - task check / off-topic check
        - feedback + verification
        - retrieved cases
        - trace log từng step của agent
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            prompt_in = gr.Textbox(
                label="IELTS Writing Task 2 Prompt",
                lines=5,
                value=default_prompt,
                placeholder="Nhập đề bài..."
            )
            essay_in = gr.Textbox(
                label="Essay",
                lines=18,
                value=default_essay,
                placeholder="Dán bài viết vào đây..."
            )

            with gr.Row():
                max_steps_in = gr.Slider(3, 10, value=6, step=1, label="Max steps")
                top_k_vector_in = gr.Slider(5, 50, value=20, step=1, label="Top-k vector")
                top_k_final_in = gr.Slider(1, 10, value=5, step=1, label="Top-k final")

            with gr.Row():
                run_btn = gr.Button("Run agent demo", variant="primary", elem_id="run-btn")
                gr.ClearButton(
                    components=[prompt_in, essay_in],
                    value="Clear inputs"
                )

            gr.Examples(
                examples=examples,
                inputs=[prompt_in, essay_in, max_steps_in, top_k_vector_in, top_k_final_in],
                label="Example"
            )

        with gr.Column(scale=1):
            summary_out = gr.Markdown()

            with gr.Tabs():
                with gr.Tab("Scores"):
                    scores_out = gr.Markdown()
                    task_check_out = gr.Markdown()

                with gr.Tab("Feedback"):
                    feedback_out = gr.Markdown()
                    verification_out = gr.Markdown()

                with gr.Tab("Retrieved cases"):
                    cases_df_out = gr.Dataframe(
                        label="Top retrieved cases",
                        interactive=False
                    )
                    cases_md_out = gr.Markdown()

                with gr.Tab("Trace log"):
                    trace_out = gr.HTML()

                with gr.Tab("Raw state"):
                    state_out = gr.JSON(label="Agent state")

    run_btn.click(
        fn=run_gradio_agent_demo,
        inputs=[prompt_in, essay_in, max_steps_in, top_k_vector_in, top_k_final_in],
        outputs=[
            summary_out,
            scores_out,
            task_check_out,
            feedback_out,
            verification_out,
            cases_df_out,
            cases_md_out,
            trace_out,
            state_out,
        ],
    )

demo.queue(max_size=8).launch(debug=True, share=True)

/tmp/ipykernel_6360/466685749.py:8: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader("gradio") is None:
/tmp/ipykernel_6360/466685749.py:467: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:
/tmp/ipykernel_6360/466685749.py:467: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 10:17:49 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
sample_prompt = """
People nowadays sleep less than they used to in the past. What do you think is the reason for this? What are the effects of this habit?
""".strip()

sample_essay = """
Nowadays, it is common that people do not spend long hours sleeping like they did in the past. This tendency is the result of several factors, which could lead to many impacts.

This phenomenon results from a host of different factors. One of them is that they have tight schedules and face pressure from study or work, while daytime is not sufficient for them to handle, which leads to them spending nighttime doing it. For example, there are increasingly high requirements for each subject that are incorporated into the curriculum such as presentations, teamwork assignments or individual homework in the Vietnamese education system. However, attending classes already takes up most of the day, the necessity of passing exams motivates students to sacrifice sleeping time in order to meet deadlines. Another important factor is the overuse of electronic devices. This is because people are addicted to many forms of entertainment on their electronic devices due to the rapid development of technology, which encourages them to stay awake late at night. As a result, bedtime is delayed, and sleep duration is significantly reduced.

Several related problems could result from this tendency. Firstly, lack of sleep can lead to serious effects for both physical and mental health. People who suffer from sleep deprivation can experience fatigue, weakened immunity, and make them undergo chronic pain related to mental and brain health problems such as anxiety and depression. Furthermore, this process in the long run can prevent productivity from efficiency, poor concentration, and thus they can not achieve goals in academic fields as well as future career paths, especially increasing the likelihood of accidents in some circumstances.

As a mentioned above, there are several factors causing less sleeping time and this could lead to several problems relating health and work productivity
""".strip()

state = run_agent_feedback_pipeline(
    prompt=sample_prompt,
    essay=sample_essay,
    df=df_retrieval,
    db_embeddings=db_embeddings,
    max_steps=6,
    top_k_vector=20,
    top_k_final=5,
    verbose=True,
)

pretty_print_agent_result(state)

In [ ]:
sample_prompt = """
Some people think that advertisements aimed at children should be banned.
To what extent do you agree or disagree?
""".strip()

sample_essay = """
It is true that social media platforms such as Facebook and Twitter have become highly popular and are increasingly replacing face-to-face contact in everyday life. Although the use of social media has several advantages, I believe that the disadvantages are far more significant.

On the one hand, there are some minor benefits to using social media instead of face-to-face communication. One possible advantage is that it makes it easier for people to stay in contact with others who live in distant regions or different countries. For example, when international students cannot return to their hometowns, they can call their parents and relatives to share their life stories and work experiences, especially during difficult periods such as the Covid-19 pandemic. Moreover, when countries need to hold conferences to discuss solutions during emergencies, social media and online communication tools can be used effectively. Another benefit is that it helps people save time. This is because some tasks do not require face-to-face interaction; for instance, students can discuss their homework online and teachers can send notifications about unexpected absences.

On the other hand, the disadvantages of replacing face-to-face contact with social media can be much more serious. One major issue is that it can make people disconnected in real life. This is because social media unintentionally creates a virtual world for each person, leading to situations in which people are physically close to one another but do not actually communicate. For example, a mother may be chatting with her friend, the father may be watching a football match, and the children may be playing games on their own devices. Another drawback is that people can become addicted to social media platforms such as Facebook and Twitter, which may result in negative effects on both health and personal development, including obesity, depression, and a lack of social skills. In fact, when people spend too much time using social media, they may become lazy, avoid going out with friends, and gradually lose important real-life communication skills.

In conclusion, despite the potential benefits of using social media, I believe that the disadvantages are much more significant.
""".strip()

state = run_agent_feedback_pipeline(
    prompt=sample_prompt,
    essay=sample_essay,
    df=df_retrieval,
    db_embeddings=db_embeddings,
    max_steps=6,
    top_k_vector=20,
    top_k_final=5,
    verbose=True,
)

pretty_print_agent_result(state)

In [ ]:
sample_prompt = """
Some people think that advertisements aimed at children should be banned.
To what extent do you agree or disagree?
""".strip()

sample_essay = """
It is sometimes believed that banning advertisements focus on aimed at children is necessary. This essay strongly agrees with is this suggestion for several reasons.

The first argument given to support my opinion is that advertisements can have some negative effects on the chidren children's mindset and behaviors. This is because, advertisements always have some unique idea, even lie ideal sometimes illogical in order to attract people to their products. While children, who have not yet enough awareness to distinguish between reality and marketing tactics, can mimic some actions and result in some bad consequences. For example, in order to introduce the benefit of their product such as strengthening bones, some milk companies depict images of people breaking walls or jumping from great heights without any issues. This makes children can follow and have some dangerous actions causing children to imitate these actions, which can lead to dangerous behaviors. This is why I believe that the government should ban advertisements aimed at children and therefore should be reduce the negative impacts of illogical advertisements on children.

Another point behind my belief is that children can be manipulated by scam advertisements and make create financial pressure on parents. The reason for this is that children can be interested in some expensive products and strongly urge parents to buy for them. This can result in the fact that parents can be tend to believe their children and don't find information to clearify itail to verify or clarify information, and thus they can lose money unfairly on a low quality product. While parents don't keep up with demand, they can get involved in criminal activities such as burglary. For these reasons, I think that prohibiting companies from advertisements aimed at children can prevent children from pressing financial exerting financial pressure on parents, thereby thereby helping people avoid deception.

In conclusion, i totally strongly agree with the idea of a ban being imposed on advertisementshat advertisements aimed at children should be banned given the aforementioned arguments.
""".strip()

state = run_agent_feedback_pipeline(
    prompt=sample_prompt,
    essay=sample_essay,
    df=df_retrieval,
    db_embeddings=db_embeddings,
    max_steps=6,
    top_k_vector=20,
    top_k_final=5,
    verbose=True,
)

pretty_print_agent_result(state)